# Collect data

In [ ]:
import pandas as pd
from utils.ols_forecasting_sanity_check import *
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.tsatools import add_trend, lagmat, lagmat2ds
from statsmodels.tools.tools import add_constant
import pelz_datasets

import warnings
warnings.filterwarnings('ignore')

In [ ]:
pd.set_option('display.max_columns', 10)

In [ ]:
output_prefix = 'data/outputs/plus1_log10_linear_imputation'
ts_data = pd.read_csv('data/outputs/plus1_log10_linear_imputation/stationary_ts_df.csv', index_col=0)
unfiltered_output_prefix = 'data/outputs/plus1_log10_linear_imputation'
max_fixed_lag = 3


In [ ]:
interpolated_ts_df = pd.read_csv('data/outputs/plus1_log10_linear_imputation/lin_interpolated_ts_df.csv', index_col=0)
interpolated_ts_df

In [ ]:
dvg2nonzero_counts = {k:v for k,v in zip(interpolated_ts_df.columns, (interpolated_ts_df>0).sum().values)}
dvg2nonzero_counts

In [ ]:
summary_df = {}
bh_summary_df = {}

for lag in range(1, max_fixed_lag):
    summary_df[lag] = pd.read_csv(f'{unfiltered_output_prefix}/gc_summary_df_lag{lag}.csv', index_col=0)
    bh_summary_df[lag] = pd.read_csv(f'{unfiltered_output_prefix}/gc_summary_df_corrected_lag{lag}.csv', index_col=0)
    #summary_df[lag] = summary_df[lag][summary_df[lag].index.isin(selected_dips)]
    #bh_summary_df[lag] = bh_summary_df[lag][bh_summary_df[lag].index.isin(selected_dips)]

In [ ]:
import matplotlib.pyplot as plt

# Extract keys and value counts
keys = summary_df.keys()
value_counts = [summary_df[key].plaque_assay_granger_label.value_counts() for key in keys]

# Convert value counts to a DataFrame for plotting
value_counts_df = pd.DataFrame(value_counts, index=keys).fillna(0)

# Plot the da 

# Functions

In [ ]:
from statsmodels.api import OLS, add_constant
from statsmodels.tsa.tsatools import lagmat2ds

def fit_autoregressive_model(x_data: pd.DataFrame, 
                                lags: int = 1, 
                                model_type: str = 'full',
                                plot: bool = False):
    """
    Fit a linear autoregressive model with variable lags.

    Parameters:
    - x_data: DataFrame with columns ['plaque_assay', 'PB2_217_2204']
    - lags: Number of lags to use
    - model_type: 'restricted' or 'full'

    Returns:
    - fitted statsmodels OLS model
    """

    assert model_type in ['restricted', 'full'], "model_type must be 'restricted' or 'full'"

    # Create lagged dataset
    dta = lagmat2ds(x_data, lags, trim="both", dropex=1)

    if model_type == 'restricted':
        # Only use lagged plaque_assay values (columns 1 to lags+1)
        X = dta[:, 1 : (lags + 1)]
    else:  # full model
        # Use lagged plaque_assay and lagged DVG (all inputs)
        X = dta[:, 1:]

    X = add_constant(X, prepend=False)
    y = dta[:, 0]  # target variable is plaque_assay at current time

    model = OLS(y, X).fit()
    if plot:
        fig, ax = plt.subplots(figsize=(8, 6))
        model_color = 'orange' if model_type == 'restricted' else 'tab:blue'
        ax.plot(model.fittedvalues, label='Fitted Values', color=model_color)
        ax.plot(y, label='Actual Values', color='black', linestyle='--')
        ax.set_title(f'Autoregressive Model Fit (lags={lags}, type={model_type})')
    return model

def step_by_step_forecast(model,
                          predicted_value: str,
                          additional_value: str,
                          initial_values: pd.DataFrame, 
                          future_dvg: pd.Series, 
                          lags: int = 1, 
                          model_type: str = 'full') -> pd.Series:
    """
    Step-by-step forecast for a linear autoregressive model.

    Parameters:
    - model: Trained statsmodels OLS model
    - predicted_value: Column name for plaque assay values
    - additional_value: Column name for DVG values
    - initial_values: DataFrame with training data (at least `lags` rows)
    - future_dvg: Series of future DVG values (must match prediction steps)
    - lags: Number of lags used in the model
    - model_type: 'restricted' or 'full'

    Returns:
    - pd.Series of predictions indexed like future_dvg
    """
    assert model_type in ['restricted', 'full'], "model_type must be 'restricted' or 'full'"
    assert len(initial_values) >= lags, "Not enough initial data for the given lag"

    # Initialize buffers
    plaque_buffer = initial_values[predicted_value].iloc[-lags:].tolist()
    if model_type == 'full':
        dvg_buffer = initial_values[additional_value].iloc[-lags:].tolist()

    predictions = []

    for t in range(len(future_dvg)):
        x_input = []

        # Add lagged plaque values
        x_input.extend(reversed(plaque_buffer))

        # If full model, add lagged DVG values
        if model_type == 'full':
            x_input.extend(reversed(dvg_buffer))

        # Add constant term (intercept) as LAST term
        x_input = x_input + [1]
                
        # Convert to 2D shape for predict
        pred = model.predict(np.array(x_input).reshape(1, -1))[0] #TODO: check size
        predictions.append(pred)

        # Update buffers
        plaque_buffer.append(pred)
        if len(plaque_buffer) > lags:
            plaque_buffer.pop(0)

        if model_type == 'full':
            dvg_buffer.append(future_dvg.iloc[t])
            if len(dvg_buffer) > lags:
                dvg_buffer.pop(0)

    return pd.Series(predictions, index=future_dvg.index)

def squared_error(y_true: pd.Series, y_pred: pd.Series) -> pd.Series:
    """
    Compute element-wise squared error between two pandas Series.

    Parameters:
    - y_true: pd.Series of true values
    - y_pred: pd.Series of predicted values

    Returns:
    - pd.Series of squared errors (same index as input)
    """
    if not isinstance(y_true, pd.Series) or not isinstance(y_pred, pd.Series):
        raise TypeError("Both y_true and y_pred must be pandas Series.")
    
    if not y_true.index.equals(y_pred.index):
        raise ValueError("Indices of y_true and y_pred must match.")

    return (y_true - y_pred) ** 2

import matplotlib.pyplot as plt

def plot_forecasts_simple(actual, restr_preds, full_preds, dvg='', lag=1, yscale='linear', ylims=None, title_suffix='', figsize=(6, 3)):
    """
    Plot actual vs. restricted and full model predictions.

    Parameters:
    - actual: Array-like of actual values (e.g., from x_test)
    - restr_preds: Restricted model predictions
    - full_preds: Full model predictions
    - lag: Lag used in the model
    - yscale: Y-axis scale (default 'symlog')
    - title_suffix: Optional string to append to title
    - figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)

    plt.plot(actual, label='Actual', linestyle='-', marker='o', color='black')
    plt.plot(restr_preds, label='Restricted Prediction', linestyle='--', marker='o', color='tab:orange')
    plt.plot(full_preds, label=f'Full Prediction\n(with {dvg})', linestyle='--', marker='o', color='tab:blue')

    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.title(f"Lag {lag}: {len(actual)}-Day Forecast: Actual vs. Predicted{title_suffix}")
    plt.xlabel("Days post infection")
    plt.ylabel("Plaque Assay")
    plt.yscale(yscale)
    
    if ylims is not None:
      plt.ylim(ylims)
    
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
import matplotlib.pyplot as plt

def plot_errors(errors_restr, errors_full, title_suffix='', yscale='linear', figsize=(6, 3),
                error_metric='Squared Error', ylims=None):
    """
    Plot element-wise squared errors for restricted and full models.

    Parameters:
    - errors_restr: Array of squared errors from restricted model
    - errors_full: Array of squared errors from full model
    - title_suffix: Optional string to append to the plot title
    - figsize: Figure size tuple
    """
    plt.figure(figsize=figsize)

    plt.plot(errors_restr, label='Restricted Model Squared Error', linestyle='--', marker='o', color='tab:orange')
    plt.plot(errors_full, label='Full Model Squared Error', linestyle='--', marker='o', color='tab:blue')

    plt.title(f'{error_metric} over time{title_suffix}')
    plt.xlabel('Days post infection')
    plt.ylabel(error_metric)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.yscale(yscale)
    if ylims is not None:
        plt.ylim(ylims)
    plt.grid(True)
    try:
      plt.tight_layout()
    except ValueError:
      # If tight_layout fails, just show the plot without it
      pass
    plt.show()



# Get random comparison data

In [ ]:
random_ts_data = pd.read_csv('data/outputs/plus1_log10_random_shuffled/stationary_ts_df.csv', index_col=0)
random_dvg_ts_data = random_ts_data[random_ts_data.columns[1:]]
random_dvg_ts_data

In [ ]:
lin_interpolated_random_ts_data = pd.read_csv('data/outputs/plus1_log10_random_shuffled/random_lin_interpolated_ts_data.csv', index_col=0)
lin_interpolated_shuffled_ts_data = pd.read_csv('data/outputs/plus1_log10_random_shuffled/shuffled_lin_interpolated_ts_data.csv', index_col=0)
dvg2nonzero_counts = dvg2nonzero_counts | {k:v for k,v in zip(lin_interpolated_random_ts_data.columns, (lin_interpolated_random_ts_data>0).sum().values)} | \
  {k:v for k,v in zip(lin_interpolated_shuffled_ts_data.columns, (lin_interpolated_shuffled_ts_data>0).sum().values)}
  
dvg2nonzero_counts

In [ ]:
np.random.seed(42)
sample_size = int(len(ts_data.columns)/3)
ts_data_random_sampled = random_dvg_ts_data[[dvg for dvg in random_dvg_ts_data.columns if 'rand' in dvg]].sample(sample_size, axis=1)
ts_data_shuffled_sampled = random_dvg_ts_data[[dvg for dvg in random_dvg_ts_data.columns if 'shuf' in dvg]].sample(sample_size, axis=1)
ts_data_shuffled_random = pd.concat([ts_data, 
                                    ts_data_shuffled_sampled,
                                    ts_data_random_sampled], axis=1)
ts_data_shuffled_random

# Functions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_mean_predictions_by_dpi(df: pd.DataFrame,
                                dpi_col: str = 'dpi',
                                restr_col: str = 'restr_preds',
                                full_col: str = 'full_preds',
                                actual_col: str = 'actual',
                                title: str = 'Mean Predictions with Error Bars by DPI',
                                xlabel: str = 'Days Post Infection (dpi)',
                                ylabel: str = 'Predicted PFU',
                                yscale: str = 'linear',
                                ylim: tuple = None,
                                figsize: tuple = (10, 6),
                                full_model_color: str = 'tab:blue',
                                restr_model_color: str = 'tab:orange',
                                actual_color: str = 'black',
                                fig: plt.Figure = None,
                                ax: plt.Axes = None):
    """
    Plots mean predictions with error bars for each dpi.

    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model predictions
    - full_col: Column name for full model predictions
    - title: Plot title
    - xlabel: Label for the x-axis
    - ylabel: Label for the y-axis
    - log_y: If True, use logarithmic scale on y-axis
    """

    # Group by dpi and compute mean and std
    grouped = df.groupby(dpi_col)[[restr_col, full_col, actual_col]].agg(['median', 'std'])

    # Flatten MultiIndex columns
    grouped.columns = ['_'.join(col) for col in grouped.columns]

    # Extract values
    dpi = grouped.index
    restr_mean = grouped[f'{restr_col}_median']
    restr_std = grouped[f'{restr_col}_std']
    full_mean = grouped[f'{full_col}_median']
    full_std = grouped[f'{full_col}_std']
    
    actual_mean = grouped[f'{actual_col}_median']
    actual_std = grouped[f'{actual_col}_std']     

    # Plot
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
    ax.errorbar(dpi, restr_mean, yerr=restr_std, fmt='o-', capsize=3, label='Restricted Prediction', color=restr_model_color)
    ax.errorbar(dpi, full_mean, yerr=full_std, fmt='s--', capsize=3, label='Full Prediction', color=full_model_color)
    ax.errorbar(dpi, actual_mean, yerr=0, fmt='D-', label='Actual Values', color=actual_color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax

def plot_median_errors_by_dpi(df: pd.DataFrame,
                        dpi_col: str = 'dpi',
                        restr_col: str = 'restr_squared_errors',
                        full_col: str = 'full_squared_errors',
                        title: str = 'Squared Errors by DPI',
                        xlabel: str = 'Days Post Infection (dpi)',
                        ylabel: str = 'Squared Error',
                        yscale: str = 'linear',
                        ylim: tuple = None,
                        full_model_color: str = 'tab:blue',
                        restr_model_color: str = 'tab:orange',
                        figsize: tuple = (10, 6),
                        fig: plt.Figure = None,
                        ax: plt.Axes = None):
    """
    Plots errors for restricted and full models by dpi. define error metric with restr_col and full_col and ylabel.

    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model squared errors
    - full_col: Column name for full model squared errors
    """
    
    # Group by dpi and compute mean and std
    grouped = df.groupby(dpi_col)[[restr_col, full_col]].agg(['median', 'std'])

    # Flatten MultiIndex columns
    grouped.columns = ['_'.join(col) for col in grouped.columns]

    # Extract values
    dpi = grouped.index
    restr_mean = grouped[f'{restr_col}_median']
    restr_std = grouped[f'{restr_col}_std']
    full_mean = grouped[f'{full_col}_median']
    full_std = grouped[f'{full_col}_std']

    # Plot
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
    ax.errorbar(dpi, restr_mean, yerr=restr_std, fmt='o-', capsize=3, label='Restricted Model Squared Error', color=restr_model_color)
    ax.errorbar(dpi, full_mean, yerr=full_std, fmt='s--', capsize=3, label='Full Model Squared Error', color=full_model_color)
    
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax
    
granger_group_colormap = {'causing' : 'tab:blue',
                          'caused': 'tab:red',
                          'bi-directional': 'tab:purple',
                          'non-related': 'gray'}

def plot_all_predictions_by_dpi(df: pd.DataFrame,
                                dpi_col: str = 'dpi',
                                restr_col: str = 'restr_preds',
                                full_col: str = 'full_preds',
                                actual_col: str = 'actual',
                                title: str = 'Mean Predictions with Error Bars by DPI',
                                xlabel: str = 'Days Post Infection (dpi)',
                                ylabel: str = 'Predicted PFU',
                                yscale: str = 'linear',
                                ylim: tuple = None,
                                figsize: tuple = (10, 6),
                                full_model_color: str = 'tab:blue',
                                alpha_col: str = None,
                                restr_model_color: str = 'tab:orange',
                                actual_color: str = 'black',
                                fig: plt.Figure = None,
                                ax: plt.Axes = None):
    """
    Plots mean predictions with error bars for each dpi.

    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model predictions
    - full_col: Column name for full model predictions
    - title: Plot title
    - xlabel: Label for the x-axis
    - ylabel: Label for the y-axis
    - log_y: If True, use logarithmic scale on y-axis
    """
    dpi = df[dpi_col]
    actual_values = df[actual_col].drop_duplicates()
    restr_values = df[restr_col].drop_duplicates()
    full_values = df[full_col]
    
    # plot
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
      
    if alpha_col is not None:
      added_to_legend = False
      max_value = df[alpha_col].max() # Get the maximum value in the group
      alphas = list(df[alpha_col] / max_value)
      for i in range(len(dpi) - 1):
        if not added_to_legend:
          ax.plot(dpi[i:i+2], full_values[i:i+2],
                  color=full_model_color, alpha=alphas[i] / 2,
                  label='Full Prediction')
          added_to_legend = True
        else:
          ax.plot(dpi[i:i+2], full_values[i:i+2],
                color=full_model_color, alpha=alphas[i] / 2)
    else:
      ax.plot(dpi, full_values, marker='.', label='Full Prediction', color=full_model_color, alpha=0.2)
    ax.plot(dpi.drop_duplicates(), actual_values, marker='D', label='Actual Values', color=actual_color, zorder=1000)
    ax.plot(dpi.drop_duplicates(), restr_values, marker='o', label='Restricted Prediction', color=restr_model_color, zorder=900)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax

def plot_all_errors_by_dpi(df: pd.DataFrame,
                        dpi_col: str = 'dpi',
                        restr_col: str = 'restr_squared_errors',
                        full_col: str = 'full_squared_errors',
                        title: str = 'Squared Errors by DPI',
                        xlabel: str = 'Days Post Infection (dpi)',
                        ylabel: str = 'Squared Error',
                        yscale: str = 'linear',
                        ylim: tuple = None,
                        full_model_color: str = 'tab:blue',
                        alpha_col: str = None,
                        restr_model_color: str = 'tab:orange',
                        figsize: tuple = (10, 6),
                        fig: plt.Figure = None,
                        ax: plt.Axes = None):
    """
    Plots errors for restricted and full models by dpi. define error metric with restr_col and full_col and ylabel.
    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model squared errors
    - full_col: Column name for full model squared errors
    """
    dpi = df[dpi_col]
    restr_values = df[restr_col].drop_duplicates()
    full_values = df[full_col]
    # plot
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
      
    if alpha_col is not None:
      added_to_legend = False
      max_value = df[alpha_col].max() # Get the maximum value in the group
      alphas = list(df[alpha_col] / max_value)
      for i in range(len(dpi) - 1):
        if not added_to_legend:
          ax.plot(dpi[i:i+2], full_values[i:i+2],
                  color=full_model_color, alpha=alphas[i] / 2,
                  label='Full Model Squared Error')
          added_to_legend = True
        else:
          ax.plot(dpi[i:i+2], full_values[i:i+2],
                  color=full_model_color, alpha=alphas[i] / 2)
    else:
      ax.plot(dpi, full_values, marker='.', label='Full Model Squared Error', color=full_model_color, alpha=0.2)
    ax.plot(dpi.drop_duplicates(), restr_values, marker='o', label='Restricted Model Squared Error', color=restr_model_color, zorder=900)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax

def plot_grouped_errors_by_dpi(df: pd.DataFrame,
                                dpi_col: str = 'dpi',
                                groups: list = ['causing', 'caused', 'bi-directional', 'non-related'],
                                restr_col: str = 'restr_squared_errors',
                                full_col: str = 'full_squared_errors',
                                title: str = 'Grouped Squared Errors by DPI',
                                xlabel: str = 'Days Post Infection (dpi)',
                                ylabel: str = 'Squared Error',
                                yscale: str = 'linear',
                                ylim: tuple = None,
                                figsize: tuple = (10, 6),
                                fig: plt.Figure = None,
                                ax: plt.Axes = None):
    """
    Plots grouped squared errors for restricted and full models by dpi.

    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model squared errors
    - full_col: Column name for full model squared errors
    """
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
    for idx, granger_label in enumerate(groups):
        group_df = df[df['label'] == granger_label]
        if group_df.empty:
            print(f"No data for group '{granger_label}'")
            continue
        
        grouped = group_df.groupby(dpi_col)[[restr_col, full_col]].agg(['median', 'std'])
        grouped.columns = ['_'.join(col) for col in grouped.columns]
        
        dpi = grouped.index
        if idx == 0:
            restr_mean = grouped[f'{restr_col}_median']
            restr_std = grouped[f'{restr_col}_std']
            ax.errorbar(dpi, restr_mean, yerr=restr_std, fmt='o-', capsize=3, label='Restricted Model Squared Error', color='tab:orange')
        
        full_mean = grouped[f'{full_col}_median']
        full_std = grouped[f'{full_col}_std']
        
        ax.errorbar(dpi, full_mean, yerr=full_std, fmt='s--', capsize=3, 
                    label=f'Full Model Squared Error ({granger_label})', 
                    color=granger_group_colormap.get(granger_label, 'gray'))
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax

def plot_grouped_mean_predictions_by_dpi(df: pd.DataFrame,
                                      dpi_col: str = 'dpi',
                                      groups: list = ['causing', 'caused', 'bi-directional', 'non-related'],
                                      restr_col: str = 'restr_preds',
                                      full_col: str = 'full_preds',
                                      actual_col: str = 'actual',
                                      title: str = 'Grouped Mean Predictions with Error Bars by DPI',
                                      xlabel: str = 'Days Post Infection (dpi)',
                                      ylabel: str = 'Predicted PFU',
                                      yscale: str = 'linear',
                                      ylim: tuple = None,
                                      figsize: tuple = (10, 6),
                                      fig: plt.Figure = None,
                                      ax: plt.Axes = None):
    """
    Plots grouped mean predictions with error bars for each dpi.
    Parameters:
    - df: DataFrame with at least [dpi_col, restr_col, full_col]
    - dpi_col: Column name for dpi values
    - restr_col: Column name for restricted model predictions
    - full_col: Column name for full model predictions
    - actual_col: Column name for actual values
    - title: Plot title
    - xlabel: Label for the x-axis
    - ylabel: Label for the y-axis
    - yscale: Y-axis scale (default 'linear')
    - ylim: Y-axis limits (default None)
    - figsize: Figure size tuple
    """
    if fig is None or ax is None:
      fig, ax = plt.subplots(figsize=figsize)
    for idx, granger_label in enumerate(groups):
        group_df = df[df['label'] == granger_label]
        if group_df.empty:
            print(f"No data for group '{granger_label}'")
            continue
        
        grouped = group_df.groupby(dpi_col)[[restr_col, full_col, actual_col]].agg(['median', 'std'])
        grouped.columns = ['_'.join(col) for col in grouped.columns]
        
        dpi = grouped.index
        full_mean = grouped[f'{full_col}_median']
        full_std = grouped[f'{full_col}_std']
        
        ax.errorbar(dpi, full_mean, yerr=full_std, fmt='s--', capsize=3, 
                    label=f'Full Prediction ({granger_label})', 
                    color=granger_group_colormap.get(granger_label, 'gray'))
        
        actual_mean = grouped[f'{actual_col}_median']
        if idx == 0:
            restr_mean = grouped[f'{restr_col}_median']
            restr_std = grouped[f'{restr_col}_std']
            ax.errorbar(dpi, restr_mean, yerr=restr_std, fmt='o-', capsize=3, label='Restricted Prediction', color='tab:orange')
            ax.errorbar(dpi, actual_mean, yerr=0, fmt='D-', label='Actual Values', color='black')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_yscale(yscale)
    ax.set_ylim(ylim)
    ax.set_title(title)
    ax.legend()
    ax.grid(True)
    fig.tight_layout()
    return fig, ax


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

pretty_names = {
    "mannwhitney_p_uncorrected": "Mann–Whitney U (uncorrected p-values)",
    "mannwhitney_p": "Mann–Whitney U (adjusted p-values)",
    "dunn_p": "Dunn’s post-hoc test (adjusted p-values)",
    "cliffs_delta": "Cliff’s Delta (effect size)",
    "rank_biserial": "Rank-biserial Correlation (effect size)",
    "epsilon_squared": "Rank Epsilon Squared (Kruskal–Wallis effect size)",
    "kendalls_w": "Kendall’s W (Friedman agreement effect size)"
}

definitions = {
    "mannwhitney_p_uncorrected": "Unadjusted chance that two specific groups differ.",
    "mannwhitney_p": "Corrected chance that two specific groups differ.",
    "dunn_p": "Chance that two groups differ, comparing all groups together.",
    "cliffs_delta": "How much one group’s values tend to be higher or lower than another’s.",
    "rank_biserial": "Strength and direction of the difference between two groups.",
    "epsilon_squared": "How much overall differences between groups explain the data.",
    "kendalls_w": "How consistent results are across repeated conditions or subjects."
}

def get_corrected_color_name(row, column='granger_score', invert=True):
    if row['label'] == 'caused':
        base_color = 'Reds'
    elif row['label'] == 'bi-directional':
        base_color = 'Purples'
    elif row['label'] == 'causing':
        base_color = 'Blues'
    elif row['label'] == 'restricted':
        base_color = 'Oranges'
    else:
        base_color = 'Greys'

    if invert == True:
        base_color += '_r'
    return base_color

def get_corrected_color(row, column='granger_score', invert=True):
    if row['label'] == 'caused':
        base_color = plt.get_cmap('Reds')
    elif row['label'] == 'bi-directional':
        base_color = plt.get_cmap('Purples')
    elif row['label'] == 'causing':
        base_color = plt.get_cmap('Blues')
    elif row['label'] == 'restricted':
        base_color = plt.get_cmap('Oranges')
    elif row['label'] in ['non-related', 'shuffled', 'bootstrapped']:
        base_color = plt.get_cmap('Greys')
    else:
      raise ValueError(f"Unrecognized label '{row['label']}' for color mapping.")

    if invert == True:
        base_color = base_color.reversed()
    return base_color(row[column])
  
def granger_color(label):
    if label == 'caused':
        return 'tab:red'
    elif label == 'bi-directional':
        return 'tab:purple'
    elif label == 'causing':
        return 'tab:blue'
    else:
        return 'gray'

def _add_sig_bar(ax, x1, x2, y, h, text, linewidth=1, fontsize=11):
    """Draw a significance bar with a bracket and centered text."""
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='k', lw=linewidth, clip_on=False)
    ax.text((x1+x2)/2, y+h, text, ha='center', va='bottom', fontsize=fontsize)

def _add_sig_bar_strip(ax_sig, x1, x2, y, h, text, linewidth=1, fontsize=11):
    """Draw a significance bar in the top strip (ax_sig uses y in [0,1])."""
    ax_sig.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='k', lw=linewidth, clip_on=False)
    ax_sig.text((x1+x2)/2, y+h, text, ha='center', va='bottom', fontsize=fontsize)

efffect_size_dct = {
    "cliffs_delta": "δ",
    "rank_biserial": "Rank-biserial Correlation",
    "epsilon_squared": "Rank Epsilon Squared",
    "kendalls_w": "Kendall’s W"
}

def make_performance_swarmplot(
        dip_forecast_summary,
        ref_performance_dct,
        performance_metric='mape',
        forecasted_value='pfu',
        title='Granger-related DI vRNAs predictive power over PFU\n',
        ylabel='Mean Absolute Percentage Error (MAPE)',
        dot_color_ref='diff_level_norm',
        inverted_colors=True,  # higher value -> darker color
        upper_border=0.05,     # fraction of y-span kept above the highest bar (kept for spacing)
        ymax=None,
        ymin=0,
        yscale='linear',
        p_val_pos=0,
        figsize=(8, 4),
        markersize=18,
        linewidth=4,
        dotsize=5,
        stats_matrix=None,          # DataFrame of p-values (rows/cols = labels)
        stat_test_name=None,        # e.g., "Mann–Whitney U"
        effect_size_matrix=None,    # DataFrame with effect sizes, same index/cols
        effect_size_name=None,      # e.g., "Cliff's d"
        alpha=0.05,                  # significance threshold
        order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
        x='label',
        plot_reference=True
    ):
    dip_forecast_summary = dip_forecast_summary.sort_values(by=dot_color_ref, ascending=inverted_colors)
    # Colors prepared elsewhere
    dip_forecast_summary['corrected_color'] = dip_forecast_summary.apply(
        get_corrected_color, args=(dot_color_ref, inverted_colors), axis=1
    )
    dip_forecast_summary['corrected_color_name'] = dip_forecast_summary.apply(
        get_corrected_color_name, args=(dot_color_ref, inverted_colors), axis=1
    )

    # --- Figure with a thin top strip for significance bars + main axis ---
    fig = plt.figure(figsize=figsize)

    if stats_matrix is not None:
      gs = GridSpec(nrows=2, ncols=1, height_ratios=[7, 14], hspace=0.0)
      ax_sig = fig.add_subplot(gs[0])   # strip for brackets
      ax     = fig.add_subplot(gs[1])   # main plot
    else:
      ax = fig.add_subplot(1, 1, 1)
      
    # force categorical hue
    dip_forecast_summary = dip_forecast_summary.copy()
    dip_forecast_summary['dvg'] = dip_forecast_summary['dvg'].astype(str)  # or .astype('category')

    palette = (dip_forecast_summary[['dvg', 'corrected_color']]
              .set_index('dvg')['corrected_color']
              .to_dict())

    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'restricted') & (dip_forecast_summary['corrected_color_name'] != 'Oranges_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'causing') & (dip_forecast_summary['corrected_color_name'] != 'Blues_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'caused') & (dip_forecast_summary['corrected_color_name'] != 'Reds_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'bi-directional') & (dip_forecast_summary['corrected_color_name'] != 'Purples_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'non-related') & (dip_forecast_summary['corrected_color_name'] != 'Greys_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'shuffled') & (dip_forecast_summary['corrected_color_name'] != 'Greys_r')]) == 0
    assert len(dip_forecast_summary[(dip_forecast_summary['label'] == 'bootstrapped') & (dip_forecast_summary['corrected_color_name'] != 'Greys_r')]) == 0
    assert len(dip_forecast_summary['dvg'].unique()) == len(dip_forecast_summary)
    # Create the swarm plot
    sns.swarmplot(
        x=x,
        y=performance_metric,
        hue='dvg',
        data=dip_forecast_summary,
        order=order,
        palette=palette,
        size=dotsize,
        ax=ax,
        legend=False,
        linewidth=0.2
    )

    # Title / labels
    fig.suptitle(f"{title}", y=0.98)
    ax.set_xlabel('Granger causality label', labelpad=10)
    ax.set_ylabel(ylabel)

    if plot_reference:
      # Reference performance line
      xmin, xmax = ax.get_xlim()
      ax.hlines(
          y=ref_performance_dct[performance_metric.upper()],
          xmin=xmin, xmax=xmax,
          label='restricted model median',
          color='#e09312', linewidth=linewidth, zorder=10
      )
      ax.legend(loc='upper left', bbox_to_anchor=(1, 1))

    # Y limits & scale
    if ymax is None:
        ymax = ref_performance_dct[performance_metric.upper()] * 3
    ax.set_ylim(ymin, ymax)
    ax.set_yscale(yscale)

    # Tick params
    ax.tick_params(top=False, labeltop=False, bottom=True, labelbottom=True)

    # Medians + sample sizes
    xs, xticklabels = ax.get_xticks(), ax.get_xticklabels()
    for values, xticklabel in zip(xs, xticklabels):
        label = xticklabel.get_text()
        med = dip_forecast_summary[dip_forecast_summary.label == label][performance_metric].median()
        ax.plot(values, med, marker='+', color='black', markersize=markersize, zorder=11)
        ax.text(values, p_val_pos, f"n={len(dip_forecast_summary[dip_forecast_summary[x] == label])}",
                ha='center', va='bottom', color='black', zorder=10)

    # --- Significance bars drawn ABOVE the plot (in ax_sig) ---
    # The strip mirrors the x-range of the main axis and uses a normalized y (0..1).

    if stats_matrix is not None:
        ax_sig.set_xlim(ax.get_xlim())
        ax_sig.set_ylim(0, 1)
        ax_sig.axis('off')
        # Ensure matrices aligned to our plotting order
        stats_mat = stats_matrix.reindex(index=order, columns=order)
        eff_mat = effect_size_matrix.reindex(index=order, columns=order) if effect_size_matrix is not None else None

        # map label -> x position
        x_pos = {lbl: x for lbl, x in zip(order, ax.get_xticks())}

        # Collect all significant pairs (upper triangle)
        pairs = []
        for i, a in enumerate(order):
            for j, b in enumerate(order):
                if j <= i:
                    continue
                p = stats_mat.loc[a, b]
                try:
                    sig = np.isfinite(p) and (p < alpha)
                except Exception:
                    sig = False
                if sig:
                    eff_txt = ""
                    if eff_mat is not None:
                        eff = eff_mat.loc[a, b]
                        if isinstance(eff, (int, float, np.floating)) and np.isfinite(eff):
                            effect_size_display_name = efffect_size_dct.get(effect_size_name, effect_size_name) if effect_size_name else None
                            eff_txt = f"{effect_size_display_name or 'effect'}={eff:.2f}"
                        else:
                            eff_txt = str(eff)
                    pairs.append((a, b, eff_txt))

        if pairs:
            # Sort by horizontal span (shortest first) so close pairs get lower levels
            pairs.sort(key=lambda t: abs(x_pos[t[0]] - x_pos[t[1]]))

            base = 0.10   # start position within the strip
            step = 0.18   # vertical spacing between stacked brackets
            h    = 0.06   # bracket height

            # Greedy level assignment to avoid horizontal overlap on the same level
            levels_spans = []  # list of (xmin, xmax) occupied spans per level
            for a, b, eff_txt in pairs:
                xa, xb = x_pos[a], x_pos[b]
                xmin_pair, xmax_pair = sorted((xa, xb))

                # find first free level where this pair does not overlap existing span
                level = None
                for li, (xmin_used, xmax_used) in enumerate(levels_spans):
                    if xmax_pair < xmin_used or xmin_pair > xmax_used:
                        level = li
                        # widen the occupied span on that level
                        levels_spans[li] = (min(xmin_used, xmin_pair), max(xmax_used, xmax_pair))
                        break
                if level is None:
                    level = len(levels_spans)
                    levels_spans.append((xmin_pair, xmax_pair))

                y_strip = base + level * step
                _add_sig_bar_strip(ax_sig, xa, xb, y_strip, h, eff_txt)

    # Leave room for the suptitle
    if stats_matrix is None:
      fig.tight_layout()
    else:
      fig.tight_layout(rect=[0, 0, 1, 1])
    return fig, ax


## stat tests

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp

# ---------- helpers ----------

def sym_matrix(levels, vals, all_levels, fill=np.nan, diag=1.0):
    """Symmetric matrix from pairwise values aligned to all_levels order."""
    n = len(all_levels)
    M = np.full((n, n), fill, dtype=float if isinstance(fill, (float, int, np.floating)) else object)
    if diag is not None:
        np.fill_diagonal(M, diag)
    k = 0
    for i in range(len(levels)):
        for j in range(i + 1, len(levels)):
            li, lj = levels[i], levels[j]
            v = vals[k]; k += 1
            if li in all_levels and lj in all_levels:
                ii, jj = all_levels.index(li), all_levels.index(lj)
                M[ii, jj] = M[jj, ii] = v
    return pd.DataFrame(M, index=all_levels, columns=all_levels)

def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta δ = P(x>y)-P(x<y), O((nx+ny)log ny)."""
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0: return np.nan
    ys = np.sort(y)
    less = np.searchsorted(ys, x, side="left")           # y < x
    greater = ny - np.searchsorted(ys, x, side="right")  # y > x
    return float((less.sum() - greater.sum()) / (nx * ny))

def epsilon_squared_kruskal(H: float, k: int, n: int) -> float:
    """Rank epsilon squared for Kruskal–Wallis."""
    if n <= k or k < 2: return np.nan
    return (H - k + 1) / (n - k)

def kendalls_w(df_long: pd.DataFrame, subject_col: str, group_col: str, value_col: str, levels: list) -> float:
    """
    Kendall's W for repeated-measures (Friedman setting).
    Uses subjects (rows) × conditions (columns) complete cases only.
    """
    if subject_col is None:
        return np.nan
    sub = df_long[[subject_col, group_col, value_col]].copy()
    sub = sub[sub[group_col].isin(levels)]
    wide = sub.pivot_table(index=subject_col, columns=group_col, values=value_col, aggfunc="mean")
    wide = wide.reindex(columns=levels)
    wide = wide.dropna(axis=0, how="any")  # complete blocks only
    n, m = wide.shape  # n subjects, m conditions
    if n < 2 or m < 2:
        return np.nan
    ranks = wide.rank(axis=1, method="average")  # rank within each subject
    Rj = ranks.sum(axis=0)                       # rank sums per condition
    Rbar = n * (m + 1) / 2.0
    S = ((Rj - Rbar) ** 2).sum()
    W = 12 * S / (m**2 * (n**3 - n))
    return float(W)

# ---------- main ----------

def pairwise_stats_tests(
    df: pd.DataFrame,
    value_col: str,
    group_col: str = "label",
    dpi_col: str = "dpi",
    mw_adjust: str = "holm",
    dunn_adjust: str = "holm",
    label_order: list = ("restricted", "causing", "bi-directional", "caused", "non-related"),
    subject_col: str = None
) -> dict:
    """
    For each dpi, returns:
      - 'mannwhitney_p_uncorrected' : pairwise MW p-value matrix
      - 'mannwhitney_p'             : pairwise MW p-value matrix (adjusted by mw_adjust)
      - 'dunn_p'                    : Dunn's post-hoc p-value matrix (adjusted by dunn_adjust)
      - 'cliffs_delta'              : pairwise Cliff's δ matrix
      - 'rank_biserial'             : pairwise rank-biserial matrix (identical to δ)
      - 'epsilon_squared'           : scalar ε² for Kruskal–Wallis (overall effect)
      - 'kendalls_w'                : scalar Kendall's W (requires subject_col; else NaN)
    All matrices are aligned to label_order; missing groups are NaN off-diagonal, 1.0 on diagonal.
    """
    results = {}
    data = df[[dpi_col, group_col, value_col] + ([subject_col] if subject_col else [])].dropna(subset=[value_col])

    for dpi_val, g in data.groupby(dpi_col):
        levels = [lab for lab in label_order if lab in g[group_col].unique()]
        if len(levels) < 2:
            continue

        # ----- pairwise Mann–Whitney + δ (and r_b) -----
        pairs = list(combinations(levels, 2))
        mw_p, deltas = [], []
        for a, b in pairs:
            x = g.loc[g[group_col] == a, value_col].to_numpy()
            y = g.loc[g[group_col] == b, value_col].to_numpy()
            if x.size == 0 or y.size == 0:
                mw_p.append(np.nan); deltas.append(np.nan)
            else:
                res = mannwhitneyu(x, y, alternative="two-sided", method="auto")
                mw_p.append(res.pvalue)
                deltas.append(cliffs_delta(x, y))

        mw_uncorr = sym_matrix(levels, mw_p, label_order, fill=np.nan, diag=1.0)
        if mw_adjust:
            valid = [p for p in mw_p if not np.isnan(p)]
            if valid:
                _, p_adj, _, _ = multipletests(valid, method=mw_adjust)
                it = iter(p_adj)
                mw_p_adj = [next(it) if not np.isnan(p) else np.nan for p in mw_p]
            else:
                mw_p_adj = mw_p
            mw_adj = sym_matrix(levels, mw_p_adj, label_order, fill=np.nan, diag=1.0)
        else:
            mw_adj = mw_uncorr.copy()

        delta_mat = sym_matrix(levels, deltas, label_order, fill=np.nan, diag=0.0)
        rank_biserial_mat = delta_mat.copy()  # r_b == δ for MW comparisons

        # ----- Dunn's post-hoc (adjusted) -----
        dunn = sp.posthoc_dunn(g, val_col=value_col, group_col=group_col, p_adjust=dunn_adjust)
        dunn = dunn.reindex(index=label_order, columns=label_order)

        # ----- Global ε² from Kruskal–Wallis -----
        arrays = [g.loc[g[group_col] == lab, value_col].to_numpy() for lab in levels]
        if all(len(a) > 0 for a in arrays) and len(arrays) >= 2:
            H, _ = kruskal(*arrays)
            eps2 = epsilon_squared_kruskal(H, k=len(arrays), n=sum(len(a) for a in arrays))
        else:
            eps2 = np.nan

        # ----- Kendall's W (optional; repeated-measures only) -----
        W = kendalls_w(g if subject_col else g.assign(__dummy=1), subject_col, group_col, value_col, levels) if subject_col else np.nan

        results[dpi_val] = {
            "mannwhitney_p_uncorrected": mw_uncorr,
            "mannwhitney_p": mw_adj,
            "dunn_p": dunn,
            "cliffs_delta": delta_mat,
            "rank_biserial": rank_biserial_mat,
            "epsilon_squared": eps2,
            "kendalls_w": W,
        }

    return results



In [ ]:
def make_forecast_analysis(summary_df, lag, train_ts_data, test_ts_data):
  labels = summary_df['plaque_assay_granger_label'].unique()
  dataframes = [summary_df[summary_df['plaque_assay_granger_label'] == label].copy() for label in labels]
  predictions_df = pd.DataFrame(columns=['dpi', 'dvg', 'label', 'ssr_chi2test_pval', 'restr_preds', 'full_preds', 'actual'])
  squared_errors_df = pd.DataFrame(columns=['dpi', 'dvg', 'label', 'ssr_chi2test_pval', 'restr_squared_errors', 'full_squared_errors'])
  exceptions = []
  for df,label in zip(dataframes,
                      labels):
      for dvg in df.index:
          x_train = train_ts_data[['plaque_assay', dvg]].copy()
          x_test = test_ts_data[['plaque_assay', dvg]].copy()
          restr_model = fit_autoregressive_model(x_train, lags=lag, model_type='restricted')
          full_model = fit_autoregressive_model(x_train, lags=lag, model_type='full')
          pval = df.loc[dvg, 'plaque_assay_ssr_chi2test']
          try:
            restr_preds = step_by_step_forecast(restr_model,
                                                predicted_value='plaque_assay',
                                                additional_value=dvg,
                                                initial_values=x_train,
                                                future_dvg=x_test[dvg],
                                                lags=lag,
                                                model_type='restricted')
            full_preds = step_by_step_forecast(full_model,
                                                predicted_value='plaque_assay',
                                                additional_value=dvg,
                                                initial_values=x_train,
                                                future_dvg=x_test[dvg],
                                                lags=lag,
                                                model_type='full')
          except Exception as e:
            #print(f"Error processing DVG {dvg} with label {label}: {e}")
            exceptions.append((dvg, label, str(e)))
            continue
          predictions_df = pd.concat([predictions_df,
                          pd.DataFrame({
                              'dpi': x_test.index,
                              'dvg': dvg,
                              'label': label,
                              'ssr_chi2test_pval': pval,
                              'restr_preds': restr_preds,
                              'full_preds': full_preds,
                              'actual': x_test['plaque_assay']
                          })],ignore_index=True)
          
          restr_squared_errors = squared_error(x_test['plaque_assay'], restr_preds)
          full_squared_errors = squared_error(x_test['plaque_assay'], full_preds)
          
          squared_errors_df = pd.concat([squared_errors_df,
                          pd.DataFrame({
                              'dpi': x_test.index,
                              'dvg': dvg,
                              'label': label,
                              'ssr_chi2test_pval': pval,
                              'restr_squared_errors': restr_squared_errors,
                              'full_squared_errors': full_squared_errors
                          })], ignore_index=True)
  return predictions_df, squared_errors_df, exceptions

def plot_forecast_analysis(predictions_df, squared_errors_df, lag=3, stats_matrix=None, stats_test_name=None, 
                          effect_size_name=None, alpha=0.001,
                          plot_path="data/plots"):
  labels = squared_errors_df['label'].unique()
  dataframes = [squared_errors_df[squared_errors_df['label'] == label].copy() for label in labels]
  
  for df,label in zip(dataframes,
                      labels):
    # fig1, axs1 = plt.subplots(figsize=(15, 4), ncols=2)
    n_dvgs = len(df['dvg'].unique())
    # plot_mean_predictions_by_dpi(predictions_df[predictions_df['label'] == label],
    #                               dpi_col='dpi',
    #                               restr_col='restr_preds',
    #                               full_col='full_preds',
    #                               actual_col='actual',
    #                               title=f'Median predictions with {n_dvgs} Granger-{label} DVGs (lag={lag})',
    #                               xlabel='Days Post Infection (dpi)',
    #                               ylabel='log10(PFU)',
    #                               yscale='linear',
    #                               full_model_color=granger_color(label),
    #                               figsize=(7,4),
    #                               ylim=(0, 10),
    #                               fig=fig1,
    #                               ax=axs1[0])
      
    # plot_median_errors_by_dpi(squared_errors_df,
    #                             dpi_col='dpi',
    #                             restr_col='restr_squared_errors',
    #                             full_col='full_squared_errors',
    #                             title=f'Median squared errors with {n_dvgs} Granger-{label} DVGs (lag={lag})',
    #                             xlabel='Days Post Infection (dpi)',
    #                             ylabel='Squared Error',
    #                             yscale='linear',
    #                             full_model_color=granger_color(label),
    #                             figsize=(7,4),
    #                             ylim=(0),
    #                             fig=fig1,
    #                             ax=axs1[1])
    
    # fig1.savefig(f"{plot_path}/median_predictions_granger_{label}_lag{lag}.png", dpi=300)
    
    # fig2, axs2 = plt.subplots(figsize=(15, 4), ncols=2)
    # plot_all_predictions_by_dpi(predictions_df[predictions_df['label'] == label],
    #                               dpi_col='dpi',
    #                               restr_col='restr_preds',
    #                               full_col='full_preds',
    #                               actual_col='actual',
    #                               title=f'All predictions with {n_dvgs} Granger-{label} DVGs (lag={lag})',
    #                               xlabel='Days Post Infection (dpi)',
    #                               ylabel='log10(PFU)',
    #                               yscale='linear',
    #                               ylim=(0, 10),
    #                               full_model_color=granger_color(label),
    #                               alpha_col='ssr_chi2test_pval',
    #                               figsize=(7,4),
    #                               fig=fig2,
    #                               ax=axs2[0])
    
    # plot_all_errors_by_dpi(squared_errors_df[squared_errors_df['label'] == label],
    #                             dpi_col='dpi',
    #                             restr_col='restr_squared_errors',
    #                             full_col='full_squared_errors',
    #                             title=f'All squared errors with {n_dvgs} Granger-{label} DVGs (lag={lag})',
    #                             xlabel='Days Post Infection (dpi)',
    #                             ylabel='Squared Error',
    #                             yscale='linear',
    #                             full_model_color=granger_color(label),
    #                             alpha_col='ssr_chi2test_pval',
    #                             figsize=(7,4),
    #                             fig=fig2,
    #                             ax=axs2[1])
    # fig2.savefig(f"{plot_path}/all_predictions_granger_{label}_lag{lag}.png", dpi=300)
  
  fig3, axs3 = plt.subplots(figsize=(18, 5), ncols=2)
  plot_grouped_mean_predictions_by_dpi(predictions_df,
                                        dpi_col='dpi',
                                        groups=labels,
                                        restr_col='restr_preds',
                                        full_col='full_preds',
                                        actual_col='actual',
                                        title=f'Grouped Mean Predictions with Error Bars by DPI (lag={lag})',
                                        xlabel='Days Post Infection (dpi)',
                                        ylabel='log10(PFU)',
                                        yscale='linear',
                                        figsize=(10, 6),
                                        ylim=(0, 10),
                                        fig=fig3,
                                        ax=axs3[0])

  plot_grouped_errors_by_dpi(squared_errors_df,
                              dpi_col='dpi',
                              groups=labels,
                              restr_col='restr_squared_errors',
                              full_col='full_squared_errors',
                              title=f'Grouped Squared Errors by DPI (lag={lag})',
                              xlabel='Days Post Infection (dpi)',
                              ylabel='Squared Error',
                              yscale='linear',
                              figsize=(10, 6),
                              fig=fig3,
                              ax=axs3[1])
  fig3.savefig(f"{plot_path}/grouped_predictions_lag{lag}.png", dpi=300)

  # For color correction later    
  # Group the DataFrame by 'label'
  grouped = squared_errors_df.groupby('label')
  squared_errors_df['group_norm_ssr_chi2test_pval'] = squared_errors_df['ssr_chi2test_pval']
  # Normalize the 'norm_ssr_chi2test' column within each group
  for label, group in grouped:
      max_value = group['group_norm_ssr_chi2test_pval'].max()  # Get the maximum value in the group
      squared_errors_df.loc[group.index, 'group_norm_ssr_chi2test_pval'] = group['group_norm_ssr_chi2test_pval'] / max_value  # Normalize

  for dpi, df in squared_errors_df.groupby('dpi'):
    restr_model_dct = {"FULL_SQUARED_ERRORS": df['restr_squared_errors'].mean()}
    swarm_fig, swarm_ax = make_performance_swarmplot(df, restr_model_dct, performance_metric='full_squared_errors', 
                              forecasted_value='pfu', title=f'Granger-related DVGs predictive power over PFU ({round(dpi,1)} dpi)\n', 
                              ylabel='Squared error', dot_color_ref='group_norm_ssr_chi2test_pval', inverted_colors=True, 
                              upper_border=-0.05, ymax=150, ymin=-10, p_val_pos=120, figsize=(12,6),
                              dotsize=5, linewidth=3, stats_matrix=stats_matrix.get(dpi).get(stats_test_name) if stats_matrix else None,
                              stat_test_name=stats_test_name, effect_size_matrix=stats_matrix.get(dpi).get(effect_size_name), 
                              effect_size_name=effect_size_name, alpha=alpha)
    swarm_fig.savefig(f"{plot_path}/swarm_performance_dpi{round(dpi,1)}_lag{lag}.png", dpi=300)
    swarm_ax.grid(True)

# Multiple dips fold and forecast

split the data into 11 folds of size 3 

In [ ]:
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 10)

In [ ]:
output_prefix

In [ ]:
n_timepoints = len(ts_data_shuffled_random)
n_splits = 8
fold_size = 3
output_prefix = output_prefix + f"_{n_splits}_folds_size{fold_size}"
test_ts_data = {}
train_ts_data = {}

fold_max_dpi = {}

ts_data_shuffled_random = ts_data_shuffled_random.fillna(0)

for i in range(n_splits):
    start = n_timepoints - (i + 1) * fold_size
    end = n_timepoints - i * fold_size
    test_ts_data[n_splits - i] = ts_data_shuffled_random.iloc[start:end]
    train_ts_data[n_splits - i] = ts_data_shuffled_random.iloc[:start]
    print(f"Fold {n_splits - i}: {test_ts_data[n_splits - i].index.tolist()}")
    fold_max_dpi[n_splits - i] = test_ts_data[n_splits - i].index.max()
    
fold_max_dpi


### functions

In [ ]:
import pandas as pd

def append_random_rows(
    summary_df: pd.DataFrame,
    ts_data_shuffled: pd.DataFrame,
    label_value: str = "shuffled",
    test_value = 1
) -> pd.DataFrame:
    """
    For each column in ts_data_shuffled, add a row to summary_df:
      - all columns ending with 'label' -> label_value
      - all columns ending with 'test'  -> test_value
      - all other columns               -> None

    If include_colname_in is provided and exists in summary_df,
    that column will store the column names from ts_data_shuffled
    (optionally prefixed with label_value).
    """
    n = ts_data_shuffled.shape[1]
    cols = summary_df.columns

    label_cols = [c for c in cols if c.endswith("label")]
    test_cols  = [c for c in cols if c.endswith("test")]

    # Create a block of n rows initialized to None
    new_rows = pd.DataFrame(None, index=range(n), columns=cols)
    new_rows.index = ts_data_shuffled.columns
    # Fill label/test columns
    if label_cols:
        new_rows.loc[:, label_cols] = label_value
    if test_cols:
        new_rows.loc[:, test_cols] = test_value
    return pd.concat([summary_df, new_rows])


In [ ]:
from scipy.stats import friedmanchisquare

def kendalls_w(df, subject_col, group_col, value_col, levels=None):
    """
    Compute Kendall's W using statsmodels' Friedman test results.
    """
    sub = df[[subject_col, group_col, value_col]].copy()
    if levels is not None:
        sub = sub[sub[group_col].isin(levels)]

    wide = sub.pivot_table(index=subject_col, columns=group_col, values=value_col, aggfunc='mean')
    if levels is not None:
        wide = wide.reindex(columns=levels)
    display(wide)
    wide = wide.dropna(axis=0, how='any')
    n, k = wide.shape
    if n < 2 or k < 2:
        return float("nan")

    # Run Friedman test
    chi2, p = friedmanchisquare(*[wide[col].values for col in wide.columns])

    # Compute Kendall’s W
    W = chi2 / (n * (k - 1))
    return W



### all

In [ ]:
summary_df_added_shuf = {}
summary_df_added_shuf_rand = {}
for lag in range(1, max_fixed_lag):
    summary_df_lag = summary_df[lag]
    summary_df_with_shuf = append_random_rows(summary_df_lag,
                                              ts_data_shuffled_random[[dvg for dvg in ts_data_shuffled_random.columns if 'shuf' in dvg]],
                                              label_value="shuffled")
    summary_df_added_shuf[lag] = summary_df_with_shuf
    summary_df_added_shuf_rand[lag] = append_random_rows(summary_df_added_shuf[lag],
                                                        ts_data_shuffled_random[[dvg for dvg in ts_data_shuffled_random.columns if 'rand' in dvg]],
                                                        label_value='bootstrapped')

In [ ]:
ts_data_shuffled_random = ts_data_shuffled_random.fillna(0)

In [ ]:
import pickle

In [ ]:
predictions_df = {}
squared_errors_df = {}
stat_test_dfs = {}
exceptions = {}

! mkdir -p {output_prefix}/forecast_plots
! mkdir -p {output_prefix}/forecast_dfs

for fold_idx in range(1, n_splits + 1):
  predictions_df[fold_idx] = {}
  squared_errors_df[fold_idx] = {}
  stat_test_dfs[fold_idx] = {}
  exceptions[fold_idx] = {}
  print(f"Processing fold {fold_idx}...")
  x_train = train_ts_data[fold_idx][['plaque_assay', 'PB2_217_2204']].copy()
  x_test = test_ts_data[fold_idx][['plaque_assay', 'PB2_217_2204']].copy()
  fig, ax_pfu = plt.subplots(figsize=(5, 3))
  ax_dvg = ax_pfu.twinx()
  ax_pfu.plot(x_train.index, x_train['plaque_assay'], marker='o', label='Training PFU', color='black')
  ax_dvg.plot(x_train.index, x_train['PB2_217_2204'], marker='o', label='Training PB2_217_2204', color='green')
  ax_pfu.plot(x_test.index, x_test['plaque_assay'], marker='o', label='Test PFU', color='black', linestyle='--')
  ax_dvg.plot(x_test.index, x_test['PB2_217_2204'], marker='o', label='Test PB2_217_2204', color='green', linestyle='--')
  ax_pfu.set_ylabel('log(PFU)')
  ax_dvg.set_ylabel('DVGs (NGS read count)')
  ax_pfu.set_xlabel('Days post infection (dpi)')
  ax_pfu.axvline(x=x_train.index[-1], color='gray', linestyle=':', label='Train/Test split')
  # legend right next to the plot
  ax_pfu.legend(loc='upper left', bbox_to_anchor=(1.2, 1))
  ax_dvg.legend(loc='upper left', bbox_to_anchor=(1.2, 0.2))
  plt.title(f'Fold {fold_idx}: Overview train/test data for an example')
  plt.show()
  
  for lag in range(1, max_fixed_lag):
    # skip if lag >= number of timepoints in training data
    if lag >= len(train_ts_data[fold_idx]):
        continue
    print(f"  Analyzing lag {lag}...")
    predictions_df[fold_idx][lag], squared_errors_df[fold_idx][lag], exceptions[fold_idx][lag] = make_forecast_analysis(
        summary_df=summary_df_added_shuf_rand[lag],
        lag=lag,
        train_ts_data=train_ts_data[fold_idx],
        test_ts_data=test_ts_data[fold_idx]
    )
    stat_test_dfs[fold_idx][lag] = pairwise_stats_tests(
        squared_errors_df[fold_idx][lag],
        value_col='full_squared_errors',
        group_col='label',
        dpi_col='dpi',
        mw_adjust='holm',
        dunn_adjust='holm',
        label_order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
        subject_col=None
    )
    
    ! mkdir -p {output_prefix}/forecast_plots/fold_{fold_idx}
    
    # plot_forecast_analysis(
    #     predictions_df=predictions_df[fold_idx][lag],
    #     squared_errors_df=squared_errors_df[fold_idx][lag],
    #     lag=lag,
    #     stats_matrix=stat_test_dfs[fold_idx][lag],
    #     stats_test_name='mannwhitney_p',
    #     effect_size_name='cliffs_delta',
    #     alpha=0.001,
    #     plot_path=f"{output_prefix}/forecast_plots/fold_{fold_idx}"
    # )
    predictions_df[fold_idx][lag].to_csv(f"{output_prefix}/forecast_dfs/predictions_fold{fold_idx}_lag{lag}.csv", index=False)
    pickle.dump(stat_test_dfs[fold_idx][lag], open(f"{output_prefix}/forecast_dfs/stat_tests_fold{fold_idx}_lag{lag}.pkl", "wb"))
    squared_errors_df[fold_idx][lag].to_csv(f"{output_prefix}/forecast_dfs/squared_errors_fold{fold_idx}_lag{lag}.csv", index=False)
    exceptions[fold_idx][lag] = pd.DataFrame(exceptions[fold_idx][lag], columns=['dvg', 'label', 'exception'])
    exceptions[fold_idx][lag].to_csv(f"{output_prefix}/forecast_dfs/exceptions_fold{fold_idx}_lag{lag}.csv", index=False)

In [ ]:
pickle.dump(predictions_df, open(f"{output_prefix}/predictions_all_folds.pkl", "wb"))
pickle.dump(squared_errors_df, open(f"{output_prefix}/squared_errors_all_folds.pkl", "wb"))
pickle.dump(stat_test_dfs, open(f"{output_prefix}/stat_tests_all_folds.pkl", "wb"))
pickle.dump(exceptions, open(f"{output_prefix}/exceptions_all_folds.pkl", "wb"))

## corrected

In [ ]:
bh_summary_df_added_shuf = {}
bh_summary_df_added_shuf_rand = {}
for lag in range(1, max_fixed_lag):
    bh_summary_df_lag = bh_summary_df[lag]
    bh_summary_df_with_shuf = append_random_rows(bh_summary_df_lag,
                                              ts_data_shuffled_random[[dvg for dvg in ts_data_shuffled_random.columns if 'shuf' in dvg]],
                                              label_value="shuffled")
    bh_summary_df_added_shuf[lag] = bh_summary_df_with_shuf
    bh_summary_df_added_shuf_rand[lag] = append_random_rows(bh_summary_df_added_shuf[lag],
                                                        ts_data_shuffled_random[[dvg for dvg in ts_data_shuffled_random.columns if 'rand' in dvg]],
                                                        label_value='bootstrapped')

In [ ]:
bh_predictions_df = {}
bh_squared_errors_df = {}
bh_stat_test_dfs = {}
bh_exceptions = {}

! mkdir -p {output_prefix}/bh_forecast_plots

for fold_idx in range(1, n_splits + 1):
  bh_predictions_df[fold_idx] = {}
  bh_squared_errors_df[fold_idx] = {}
  bh_stat_test_dfs[fold_idx] = {}
  bh_exceptions[fold_idx] = {}
  print(f"Processing BH fold {fold_idx}...")
  x_train = train_ts_data[fold_idx][['plaque_assay', 'PB2_217_2204']].copy()
  x_test = test_ts_data[fold_idx][['plaque_assay', 'PB2_217_2204']].copy()
  fig, ax_pfu = plt.subplots(figsize=(5, 3))
  ax_dvg = ax_pfu.twinx()
  ax_pfu.plot(x_train.index, x_train['plaque_assay'], marker='o', label='Training PFU', color='black')
  ax_dvg.plot(x_train.index, x_train['PB2_217_2204'], marker='o', label='Training PB2_217_2204', color='green')
  ax_pfu.plot(x_test.index, x_test['plaque_assay'], marker='o', label='Test PFU', color='black', linestyle='--')
  ax_dvg.plot(x_test.index, x_test['PB2_217_2204'], marker='o', label='Test PB2_217_2204', color='green', linestyle='--')
  ax_pfu.set_ylabel('log(PFU)')
  ax_dvg.set_ylabel('DVGs (NGS read count)')
  ax_pfu.set_xlabel('Days post infection (dpi)')
  ax_pfu.axvline(x=x_train.index[-1], color='gray', linestyle=':', label='Train/Test split')
  # legend right next to the plot
  ax_pfu.legend(loc='upper left', bbox_to_anchor=(1.2, 1))
  ax_dvg.legend(loc='upper left', bbox_to_anchor=(1.2, 0.2))
  plt.title(f'Fold {fold_idx}: Overview train/test data for an example')
  plt.show()
  
  for lag in range(1, max_fixed_lag):
    # skip if lag >= number of timepoints in training data
    if lag >= len(train_ts_data[fold_idx]):
        continue
    print(f"  Analyzing lag {lag}...")
    bh_predictions_df[fold_idx][lag], bh_squared_errors_df[fold_idx][lag], bh_exceptions[fold_idx][lag] = make_forecast_analysis(
        summary_df=bh_summary_df_added_shuf_rand[lag],
        lag=lag,
        train_ts_data=train_ts_data[fold_idx],
        test_ts_data=test_ts_data[fold_idx]
    )
    bh_stat_test_dfs[fold_idx][lag] = pairwise_stats_tests(
        bh_squared_errors_df[fold_idx][lag],
        value_col='full_squared_errors',
        group_col='label',
        dpi_col='dpi',
        mw_adjust='holm',
        dunn_adjust='holm',
        label_order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
        subject_col=None
    )
    
    ! mkdir -p {output_prefix}/bh_forecast_plots/fold_{fold_idx}
    
    # plot_forecast_analysis(
    #     predictions_df=bh_predictions_df[fold_idx][lag],
    #     squared_errors_df=bh_squared_errors_df[fold_idx][lag],
    #     lag=lag,
    #     stats_matrix=bh_stat_test_dfs[fold_idx][lag],
    #     stats_test_name='mannwhitney_p',
    #     effect_size_name='cliffs_delta',
    #     alpha=0.001,
    #     plot_path=f"{output_prefix}/bh_forecast_plots/fold_{fold_idx}"
    # )
    
pickle.dump(bh_predictions_df, open(f"{output_prefix}/bh_predictions_all_folds.pkl", "wb"))
pickle.dump(bh_squared_errors_df, open(f"{output_prefix}/bh_squared_errors_all_folds.pkl", "wb"))
pickle.dump(bh_stat_test_dfs, open(f"{output_prefix}/bh_stat_tests_all_folds.pkl", "wb"))
pickle.dump(bh_exceptions, open(f"{output_prefix}/bh_exceptions_all_folds.pkl", "wb"))

# summarize folds

In [ ]:
predictions_df = pickle.load(open(f"{output_prefix}/predictions_all_folds.pkl", "rb"))
squared_errors_df = pickle.load(open(f"{output_prefix}/squared_errors_all_folds.pkl", "rb"))
stat_test_dfs = pickle.load(open(f"{output_prefix}/stat_tests_all_folds.pkl", "rb"))
exceptions = pickle.load(open(f"{output_prefix}/exceptions_all_folds.pkl", "rb"))

bh_predictions_df = pickle.load(open(f"{output_prefix}/bh_predictions_all_folds.pkl", "rb"))
bh_squared_errors_df = pickle.load(open(f"{output_prefix}/bh_squared_errors_all_folds.pkl", "rb"))
bh_stat_test_dfs = pickle.load(open(f"{output_prefix}/bh_stat_tests_all_folds.pkl", "rb"))
bh_exceptions = pickle.load(open(f"{output_prefix}/bh_exceptions_all_folds.pkl", "rb"))

In [ ]:
mae_df = {}

for fold_idx in range(1, n_splits + 1):
  mae_df[fold_idx] = {}
  for lag in range(1, max_fixed_lag):
    if lag >= len(train_ts_data[fold_idx]):
        continue
    df = predictions_df[fold_idx][lag].copy()
    df['restr_mae'] = np.abs(df['actual'] - df['restr_preds'])
    df['full_mae'] = np.abs(df['actual'] - df['full_preds'])
    mae_df[fold_idx][lag] = df[['dpi', 'dvg', 'label', 'ssr_chi2test_pval','restr_mae', 'full_mae']]
    

In [ ]:
summarized_fold_predictions = {}
summarized_fold_squared_errors = {}
summarized_fold_maes = {}
df_counts = {}

for lag in range(1, max_fixed_lag):
    summarized_fold_predictions[lag] = pd.concat(
        [predictions_df.get(fold_idx, {}).get(lag, pd.DataFrame()) for fold_idx in range(1, n_splits + 1)],
        ignore_index=True
    )
    summarized_fold_squared_errors[lag] = pd.concat(
        [squared_errors_df.get(fold_idx, {}).get(lag, pd.DataFrame()) for fold_idx in range(1, n_splits + 1)],
        ignore_index=True
    )
    summarized_fold_squared_errors[lag]['error_diff'] = summarized_fold_squared_errors[lag]['restr_squared_errors'] - summarized_fold_squared_errors[lag]['full_squared_errors']
    summarized_fold_squared_errors[lag] = summarized_fold_squared_errors[lag].sort_values(by='error_diff', ascending=False)
    df_counts[lag] = (
        summarized_fold_squared_errors[lag][summarized_fold_squared_errors[lag]["error_diff"] > 0]
        .groupby("dvg")
        .size()
        .reset_index(name="num_positive_error_diff")
    )
    
    summarized_fold_maes[lag] = pd.concat(
        [mae_df.get(fold_idx, {}).get(lag, pd.DataFrame()) for fold_idx in range(1, n_splits + 1)],
        ignore_index=True
    )
    
    summarized_fold_maes[lag]['mae_diff'] = summarized_fold_maes[lag]['restr_mae'] - summarized_fold_maes[lag]['full_mae']
    summarized_fold_maes[lag] = summarized_fold_maes[lag].sort_values(by='mae_diff', ascending=False)
    
    df_counts[lag].sort_values(by='num_positive_error_diff', ascending=False)
    display(df_counts[lag].num_positive_error_diff.value_counts().sort_index(ascending=False))
label_order = ['causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped']

## plot average by timepoint

In [ ]:
plt.rcParams.update({'font.size': 22})

In [ ]:
label_order = ['causing', 'bi-directional', 'caused', 'non-related', 'shuffled']

fold_max_list = list(fold_max_dpi.values())

for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag}...")
  label_counts = summarized_fold_predictions[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 8), nrows=1, ncols=len(label_order), sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_predictions[lag][['dpi', 'restr_preds']].drop_duplicates()
  for label, df in summarized_fold_predictions[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    # actual values
    sns.lineplot(
      x='dpi',
      y='actual',
      color='black',
      data=df,
      ax=axs[ax_index],
      marker='s',
      alpha=0.5,
      label="Actual"
    )
    
    sns.lineplot(
    x='dpi',
    y='full_preds',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_preds',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('log10(PFU/mL+1)')
    axs[ax_index].set_yscale('linear')
    axs[ax_index].set_ylim(-1, 10)
      
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=axs[ax_index].get_ylim()[0], ymax=axs[ax_index].get_ylim()[1], colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'Predictions Full vs restricted model (Lag={lag})') 
  fig.tight_layout()
  fig.savefig(f"{output_prefix}/forecast_plots/summary_predictions_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
bh_corrected_label_dct = bh_squared_errors_df[1][1].set_index('dvg')['label'].to_dict()
fold_max_list = list(fold_max_dpi.values())
label_order = ['causing', 'bi-directional', 'caused', 'non-related', 'shuffled']

summarized_fold_predictions_bh = {}

for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag}...")
  summarized_fold_predictions_bh[lag] = summarized_fold_predictions[lag].copy()
  summarized_fold_predictions_bh[lag]['label'] = summarized_fold_predictions_bh[lag]['dvg'].map(bh_corrected_label_dct)
  label_counts = summarized_fold_predictions_bh[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 8), nrows=1, ncols=len(label_order), sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_predictions_bh[lag][['dpi', 'restr_preds']].drop_duplicates()
  for label, df in summarized_fold_predictions_bh[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    # actual values
    sns.lineplot(
      x='dpi',
      y='actual',
      color='black',
      data=df,
      ax=axs[ax_index],
      marker='s',
      alpha=0.5,
      label="Actual"
    )
    
    sns.lineplot(
    x='dpi',
    y='full_preds',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_preds',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('log10(PFU/mL+1)')
    axs[ax_index].set_yscale('linear')
    axs[ax_index].set_ylim(-1, 10)
      
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=axs[ax_index].get_ylim()[0], ymax=axs[ax_index].get_ylim()[1], colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'Predictions Full vs restricted model with corrected labels(Lag={lag})') 
  fig.tight_layout()
  fig.savefig(f"{output_prefix}/forecast_plots/summary_predictions_bh_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
bh_corrected_label_dct = bh_squared_errors_df[1][1].set_index('dvg')['label'].to_dict()

fold_max_list = list(fold_max_dpi.values())
summarized_fold_maes_bh = {}
for lag in range(1, max_fixed_lag):
  summarized_fold_maes_bh[lag] = summarized_fold_maes[lag].copy()
  summarized_fold_maes_bh[lag]['label'] = summarized_fold_maes_bh[lag]['dvg'].map(bh_corrected_label_dct)
  ymaxs = []
  print(f"Plotting lag {lag}...")
  label_counts = summarized_fold_maes_bh[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 8), nrows=1, ncols=len(label_order), sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_maes_bh[lag][['dpi', 'restr_mae']].drop_duplicates()
  for label, df in summarized_fold_maes_bh[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    
    sns.lineplot(
    x='dpi',
    y='full_mae',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_mae',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('mae')
    axs[ax_index].set_yscale('linear')
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
    
    ymaxs.append(axs[ax_index].get_ylim()[1])
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=0, ymax=max(ymaxs), colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'DVGs and how much they improved compared to restricted model(Lag={lag})') 
  fig.tight_layout()
  fig.savefig(f"{output_prefix}/forecast_plots/summary_maes_bh_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
fold_max_list = list(fold_max_dpi.values())

summarized_fold_squared_errors_bh = {}
for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag}...")
  summarized_fold_squared_errors_bh[lag] = summarized_fold_squared_errors[lag].copy()
  summarized_fold_squared_errors_bh[lag]['label'] = summarized_fold_squared_errors_bh[lag]['dvg'].map(bh_corrected_label_dct)
  label_counts = summarized_fold_squared_errors_bh[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 4), nrows=1, ncols=len(label_order), sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_squared_errors_bh[lag][['dpi', 'restr_squared_errors']].drop_duplicates()
  for label, df in summarized_fold_squared_errors_bh[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    
    sns.lineplot(
    x='dpi',
    y='full_squared_errors',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_squared_errors',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('Squared Errors')
    axs[ax_index].set_yscale('log')
      
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=axs[ax_index].get_ylim()[0], ymax=axs[ax_index].get_ylim()[1], colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'DVGs and how much they improved compared to restricted model(Lag={lag})') 
  fig.savefig(f"{output_prefix}/forecast_plots/summary_squared_errors_bh_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
summarized_fold_squared_errors[1].drop_duplicates(subset=['dpi', 'restr_squared_errors']).sort_values(by='dpi')

In [ ]:
for lag in range(1, max_fixed_lag):
  restr_df = summarized_fold_predictions[lag].drop_duplicates(subset=['dpi', 'restr_preds']).sort_values(by='dpi').copy()
  restr_df['label'] = 'restricted'
  restr_df['full_preds'] = restr_df['restr_preds']
  restr_df['ssr_chi2test_pval'] = restr_df['dpi']/restr_df['dpi'].max()
  restr_df['dvg'] = 'restr_' + restr_df['dpi'].astype(str)
  summarized_fold_predictions[lag] = pd.concat([summarized_fold_predictions[lag], restr_df], ignore_index=True)
  
  restr_error_df = summarized_fold_squared_errors[lag].drop_duplicates(subset=['dpi', 'restr_squared_errors']).sort_values(by='dpi').copy()
  restr_error_df['full_squared_errors'] = restr_error_df['restr_squared_errors']
  restr_error_df['ssr_chi2test_pval'] = restr_error_df['dpi']/restr_error_df['dpi'].max()
  restr_error_df['dvg'] = 'restr_' + restr_error_df['dpi'].astype(str)
  restr_error_df['label'] = 'restricted'
  summarized_fold_squared_errors[lag] = pd.concat([summarized_fold_squared_errors[lag], restr_error_df], ignore_index=True)
  
  restr_mae_df = summarized_fold_maes[lag].drop_duplicates(subset=['dpi', 'restr_mae']).sort_values(by='dpi').copy()
  restr_mae_df['full_mae'] = restr_mae_df['restr_mae']
  restr_mae_df['ssr_chi2test_pval'] = restr_mae_df['dpi']/restr_mae_df['dpi'].max()
  restr_mae_df['dvg'] = 'restr_' + restr_mae_df['dpi'].astype(str)
  restr_mae_df['label'] = 'restricted'
  summarized_fold_maes[lag] = pd.concat([summarized_fold_maes[lag], restr_mae_df], ignore_index=True)

In [ ]:
fold_max_list = list(fold_max_dpi.values())

for lag in range(1, max_fixed_lag):
  print(f"Plotting lag {lag}...")
  label_counts = summarized_fold_squared_errors[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 4), nrows=1, ncols=6, sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_squared_errors[lag][['dpi', 'restr_squared_errors']].drop_duplicates()
  for label, df in summarized_fold_squared_errors[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    
    sns.lineplot(
    x='dpi',
    y='full_squared_errors',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_squared_errors',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('Squared Errors')
    axs[ax_index].set_yscale('log')
      
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=axs[ax_index].get_ylim()[0], ymax=axs[ax_index].get_ylim()[1], colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'DVGs and how much they improved compared to restricted model(Lag={lag})') 
  fig.savefig(f"{output_prefix}/forecast_plots/summary_squared_errors_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
fold_max_list = list(fold_max_dpi.values())

for lag in range(1, max_fixed_lag):
  ymaxs = []
  print(f"Plotting lag {lag}...")
  label_counts = summarized_fold_maes[lag]['label'].value_counts()
  fig, axs = plt.subplots(figsize=(30, 4), nrows=1, ncols=6, sharey=True)
  axs = axs.flatten()
  
  restr_values = summarized_fold_maes[lag][['dpi', 'restr_mae']].drop_duplicates()
  for label, df in summarized_fold_maes[lag].groupby('label'):
    if label not in label_order:
        continue
    ax_index = label_order.index(label)
    
    sns.lineplot(
    x='dpi',
    y='full_mae',
    color=granger_color(label),
    data=df,
    ax=axs[ax_index],
    marker='o',
    alpha=0.8,
    label="Full model"
  )

    sns.lineplot(
      x='dpi',
      y='restr_mae',
      marker='D',
      ax=axs[ax_index],
      data=restr_values,
      color='tab:orange',
      label='Restricted Model')
      
    axs[ax_index].set_title(f"Label: {label} ({len(df['dvg'].unique())})")
    axs[ax_index].set_xlabel('DPI')
    axs[ax_index].set_ylabel('mae')
    axs[ax_index].set_yscale('linear')
    axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
    
    ymaxs.append(axs[ax_index].get_ylim()[1])
  for label in label_order:
    ax_index = label_order.index(label)
    axs[ax_index].vlines(fold_max_list, ymin=0, ymax=max(ymaxs), colors='gray', linestyles='dashed', alpha=0.5)
  fig.suptitle(f'DVGs and how much they improved compared to restricted model(Lag={lag})') 
  fig.savefig(f"{output_prefix}/forecast_plots/summary_maes_by_label_lag{lag}.png", dpi=300, bbox_inches='tight')


In [ ]:
output_prefix

In [ ]:
top_mae_dvgs = {}
for lag in range(1, 2):
    top_mae_dvgs[lag] = summarized_fold_maes[lag][~summarized_fold_maes[lag]['dvg'].str.contains('restr')].groupby('dvg')['full_mae'].median().sort_values(ascending=True).head(50)

sorted_by_median_mae = summarized_fold_maes[1][~summarized_fold_maes[1]['dvg'].str.contains('restr')].groupby('dvg')['full_mae'].median().sort_values(ascending=True)



label_order = ['causing',
              'bi-directional',
              'caused',
              'non-related',
              'shuffled']

fold_max_list = list(fold_max_dpi.values())

top_dip_index_lst = [1,2,3,4]
for dip_idx in top_dip_index_lst:
  tmp_df = summarized_fold_maes[1][summarized_fold_maes[1]['dvg'].isin(top_mae_dvgs[1].index)].set_index('dvg').loc[top_mae_dvgs[1].index].reset_index()

  df = summarized_fold_maes[1][summarized_fold_maes[1]['dvg'].isin(sorted_by_median_mae.index)].set_index('dvg').loc[sorted_by_median_mae.index].reset_index()
  selected_dvg_dct_lag1 = {'bi-directional':tmp_df[tmp_df.label=='bi-directional'].drop_duplicates(subset=['dvg']).iloc[dip_idx]['dvg'],
                          'causing':tmp_df[tmp_df.label=='causing'].drop_duplicates(subset=['dvg']).iloc[dip_idx]['dvg'],
                          'caused':df[df.label=='caused'].iloc[3000]['dvg'],
                          'non-related':df[df.label=='non-related'].iloc[3000]['dvg'],
                          'shuffled':df[df.label=='shuffled'].iloc[3000]['dvg']}

  for lag in range(1, 2):
    print(f"Plotting lag {lag}...")
    label_counts = summarized_fold_predictions[lag]['label'].value_counts()
    fig, axs = plt.subplots(figsize=(35, 9), nrows=1, ncols=len(label_order), sharey=True)
    axs = axs.flatten()
    
    restr_values = summarized_fold_predictions[lag][['dpi', 'restr_preds']].drop_duplicates()
    for label, df in summarized_fold_predictions[lag].groupby('label'):
      if label not in label_order:
          continue
      dvg = selected_dvg_dct_lag1.get(label, '')
      ax_index = label_order.index(label)
      # actual values
      sns.lineplot(
        x='dpi',
        y='actual',
        color='black',
        data=df,
        ax=axs[ax_index],
        marker='s',
        alpha=0.5,
        label="Actual"
      )
      
      sns.lineplot(
      x='dpi',
      y='full_preds',
      color=granger_color(label),
      data=df[df['dvg'] == dvg],
      ax=axs[ax_index],
      marker='o',
      alpha=0.8,
      label=f"Full model with {dvg}"
    )

      sns.lineplot(
        x='dpi',
        y='restr_preds',
        marker='D',
        ax=axs[ax_index],
        data=restr_values,
        color='tab:orange',
        label='Restricted Model')
        
      axs[ax_index].set_title(f"Label: {label}")
      axs[ax_index].set_xlabel('DPI')
      axs[ax_index].set_ylabel('log10(PFU/mL+1)')
      axs[ax_index].set_yscale('linear')
      axs[ax_index].set_ylim(-1, 10)
        
      axs[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
    for label in label_order:
      ax_index = label_order.index(label)
      axs[ax_index].vlines(fold_max_list, ymin=axs[ax_index].get_ylim()[0], ymax=axs[ax_index].get_ylim()[1], colors='gray', linestyles='dashed', alpha=0.5)
    fig.suptitle(f'Predictions Full vs restricted model for selected DVGs (Lag={lag})') 
    fig.tight_layout()
    
    

    ymaxs = []
    print(f"Plotting lag {lag}...")
    label_counts = summarized_fold_maes[lag]['label'].value_counts()
    fig2, axs2 = plt.subplots(figsize=(35, 9), nrows=1, ncols=5, sharey=True)
    axs2 = axs2.flatten()
    
    restr_values = summarized_fold_maes[lag][['dpi', 'restr_mae']].drop_duplicates()
    for label, df in summarized_fold_maes[lag].groupby('label'):
      if label not in label_order:
          continue
      ax_index = label_order.index(label)
      dvg = selected_dvg_dct_lag1.get(label, '')
      
      sns.lineplot(
      x='dpi',
      y='full_mae',
      color=granger_color(label),
      data=df[df['dvg'] == dvg],
      ax=axs2[ax_index],
      marker='o',
      alpha=0.8,
      label=f"Full model with {dvg}"
    )

      sns.lineplot(
        x='dpi',
        y='restr_mae',
        marker='D',
        ax=axs2[ax_index],
        data=restr_values,
        color='tab:orange',
        label='Restricted Model')
        
      axs2[ax_index].set_title(f"Label: {label}")
      axs2[ax_index].set_xlabel('DPI')
      axs2[ax_index].set_ylabel('MAE')
      axs2[ax_index].set_yscale('linear')
      axs2[ax_index].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)
      
      ymaxs.append(axs2[ax_index].get_ylim()[1])
    for label in label_order:
      ax_index = label_order.index(label)
      axs2[ax_index].vlines(fold_max_list, ymin=0, ymax=max(ymaxs), colors='gray', linestyles='dashed', alpha=0.5)
    fig2.tight_layout()
    #fig2.suptitle(f'Selected DVGs and how much they improved compared to restricted model(Lag={lag})') 


In [ ]:
plt.rcParams.update({'font.size': 24})
top_mae_dvgs = {}
for lag in range(1, 2):
    top_mae_dvgs[lag] = summarized_fold_maes[lag][~summarized_fold_maes[lag]['dvg'].str.contains('restr')].groupby('dvg')['full_mae'].median().sort_values(ascending=True).head(50)

sorted_by_median_mae = summarized_fold_maes[1][~summarized_fold_maes[1]['dvg'].str.contains('restr')].groupby('dvg')['full_mae'].median().sort_values(ascending=True)

label_order = ['causing',
'bi-directional',
'caused',
'non-related',
'shuffled']

fold_max_list = list(fold_max_dpi.values())

top_dip_index_lst = [0,1,2]
for dip_idx in top_dip_index_lst:
  tmp_df = summarized_fold_maes[1][summarized_fold_maes[1]['dvg'].isin(top_mae_dvgs[1].index)].set_index('dvg').loc[top_mae_dvgs[1].index].reset_index()

  df = summarized_fold_maes[1][summarized_fold_maes[1]['dvg'].isin(sorted_by_median_mae.index)].set_index('dvg').loc[sorted_by_median_mae.index].reset_index()
  selected_dvg_dct_lag1 = {'bi-directional':tmp_df[tmp_df.label=='bi-directional'].drop_duplicates(subset=['dvg']).iloc[dip_idx]['dvg'],
                          'causing':tmp_df[tmp_df.label=='causing'].drop_duplicates(subset=['dvg']).iloc[dip_idx]['dvg'],
                          'caused':df[df.label=='caused'].iloc[3000]['dvg'],
                          'non-related':df[df.label=='non-related'].iloc[3000]['dvg'],
                          'shuffled':df[df.label=='shuffled'].iloc[3000]['dvg']}

  for lag in range(1, 2):
    print(f"Plotting lag {lag}...")
    label_counts = summarized_fold_predictions[lag]['label'].value_counts()
    
    # Create single figure with 5 rows and 2 columns
    # Use GridSpec for more control over spacing
    from matplotlib.gridspec import GridSpec
    
    fig = plt.figure(figsize=(21, 28))
    gs = GridSpec(5, 2, figure=fig, hspace=0.35, wspace=0.25, 
                  left=0.08, right=0.78, top=0.97, bottom=0.03)
    
    axs = []
    for i in range(5):
        row = []
        for j in range(2):
            ax = fig.add_subplot(gs[i, j])
            row.append(ax)
        axs.append(row)
    
    restr_pred_values = summarized_fold_predictions[lag][['dpi', 'restr_preds']].drop_duplicates()
    restr_mae_values = summarized_fold_maes[lag][['dpi', 'restr_mae']].drop_duplicates()
    
    ymaxs = []
    
    for label, pred_df in summarized_fold_predictions[lag].groupby('label'):
      if label not in label_order:
          continue
      row_index = label_order.index(label)
      dvg = selected_dvg_dct_lag1.get(label, '')
      
      # Left column (predictions) - column 0
      ax_left = axs[row_index][0]
      
      # Actual values
      sns.lineplot(
        x='dpi',
        y='actual',
        color='black',
        data=pred_df,
        ax=ax_left,
        marker='s',
        alpha=1,
        linewidth=3,
        label="Actual PFU/mL"
      )
      
      # Full model predictions
      sns.lineplot(
        x='dpi',
        y='full_preds',
        color=granger_color(label),
        data=pred_df[pred_df['dvg'] == dvg],
        ax=ax_left,
        marker='o',
        alpha=1,
        linewidth=3,
        label=f"Full model with\n{dvg}"
      )

      # Restricted model predictions
      sns.lineplot(
        x='dpi',
        y='restr_preds',
        marker='D',
        ax=ax_left,
        data=restr_pred_values,
        color='tab:orange',
        linewidth=3,
        label='Restricted Model')
        
      ax_left.set_title(f"{label}")
      ax_left.set_xlabel('time post infection (days)')
      ax_left.set_ylabel('log10(PFU/mL+1)')
      ax_left.set_yscale('linear')
      ax_left.set_ylim(-1, 10)
      ax_left.vlines(fold_max_list, ymin=ax_left.get_ylim()[0], ymax=ax_left.get_ylim()[1], 
                    colors='gray', linestyles='dashed', alpha=0.5)
      # Remove legend from left plot
      ax_left.get_legend().remove()
    
    # Right column (MAE) - column 1
    for label, mae_df in summarized_fold_maes[lag].groupby('label'):
      if label not in label_order:
          continue
      row_index = label_order.index(label)
      dvg = selected_dvg_dct_lag1.get(label, '')
      
      ax_right = axs[row_index][1]
      
      # Full model MAE
      sns.lineplot(
        x='dpi',
        y='full_mae',
        color=granger_color(label),
        data=mae_df[mae_df['dvg'] == dvg],
        ax=ax_right,
        marker='o',
        alpha=1,
        linewidth=3,
        label=f"Full model with\n{dvg}"
      )

      # Restricted model MAE
      sns.lineplot(
        x='dpi',
        y='restr_mae',
        marker='D',
        ax=ax_right,
        data=restr_mae_values,
        color='tab:orange',
        linewidth=3,
        label='Restricted Model')
        
      ax_right.set_title(f"{label}")
      ax_right.set_xlabel('Time post infection (days)')
      ax_right.set_ylabel('MAE')
      ax_right.set_yscale('linear')
      # Remove legend from right plot
      ax_right.get_legend().remove()
      
      ymaxs.append(ax_right.get_ylim()[1])
    
    # Add vertical lines to all right column plots
    for row_index in range(len(label_order)):
      axs[row_index][1].vlines(fold_max_list, ymin=0, ymax=max(ymaxs), 
                              colors='gray', linestyles='dashed', alpha=0.5)
    
    # Create shared legend per row - positioned to the right of the plots
    for row_index, label in enumerate(label_order):
      dvg = selected_dvg_dct_lag1.get(label, '')
      
      # Collect handles and labels from both axes in this row
      handles_left, labels_left = axs[row_index][0].get_legend_handles_labels()
      handles_right, labels_right = axs[row_index][1].get_legend_handles_labels()
      
      # Combine and deduplicate (keep order, remove duplicates)
      all_handles = handles_left + handles_right
      all_labels = labels_left + labels_right
      
      # Remove duplicates while preserving order
      unique_labels = []
      unique_handles = []
      for handle, lbl in zip(all_handles, all_labels):
        if lbl not in unique_labels:
          unique_labels.append(lbl)
          unique_handles.append(handle)
      
      # Get the position of the right axis for this row
      pos_right = axs[row_index][1].get_position()
      
      # Position legend to the right of the plots, vertically centered with the row
      legend_x = pos_right.x1 + 0.02
      legend_y = (pos_right.y0 + pos_right.y1) / 2
      
      # Add legend to the right of both subplots in this row
      fig.legend(unique_handles, unique_labels, 
                loc='center left',
                bbox_to_anchor=(legend_x, legend_y),
                bbox_transform=fig.transFigure,
                ncol=1,
                frameon=True)
    
    fig.suptitle(f'Predictions and MAE: Full vs Restricted Model for selected DVGs (Lag={lag})', y=1.02)
    fig.tight_layout()

## Plot by average over all timepoints

In [ ]:
summarized_fold_squared_errors[1].drop_duplicates(subset=['dvg', 'label']).value_counts('label')

In [ ]:
summarized_fold_maes[1].drop_duplicates(subset=['dvg', 'label']).value_counts('label')

In [ ]:
summarized_fold_squared_errors[1].groupby(['dvg','label']).agg({'label': 'first',
                                                      'ssr_chi2test_pval': 'first',
                                                      'dpi': 'count',
                                                      'full_squared_errors': ['mean', 'median', 'std', lambda x: (x - x.mean()).abs().mean()],
                                                      'restr_squared_errors': ['mean', 'median', 'std', lambda x: (x - x.mean()).abs().mean()],
                                                      'error_diff': ['mean', 'std']
                                                      }).label.value_counts()

summarized_fold_maes[1].groupby(['dvg','label']).agg({'label': 'first',
                                                      'ssr_chi2test_pval': 'first',
                                                      'dpi': 'count',
                                                      'full_mae': ['mean', 'median', 'std', lambda x: (x - x.mean()).abs().mean()],
                                                      'restr_mae': ['mean', 'median', 'std', lambda x: (x - x.mean()).abs().mean()],
                                                      'mae_diff': ['median', lambda x: (x - x.mean()).abs().mean()]
                                                      }).label.value_counts()

In [ ]:
avg_summarized_squared_errors = {}
median_summarized_squared_errors = {}
mean_stats_tests_dfs = {}
median_stats_tests_dfs = {}
avg_summarized_maes = {}
median_summarized_maes = {}
mean_mae_stats_tests_dfs = {}
median_mae_stats_tests_dfs = {}

for lag in range(1, max_fixed_lag):
  summarized_fold_squared_errors[lag]['grouping_id'] = summarized_fold_squared_errors[lag]['dvg'].apply(lambda x: x.split('_')[0] if x.startswith('restr_') else x)
  avg_summarized_squared_errors[lag] = summarized_fold_squared_errors[lag].groupby(['grouping_id']).agg({
                                                      'dvg': 'first',
                                                      'label': 'first',
                                                      'ssr_chi2test_pval': 'first',
                                                      'dpi': 'count',
                                                      'full_squared_errors': ['mean', 'median', 'std', lambda x: (x - x.median()).abs().median()],
                                                      'restr_squared_errors': ['mean', 'median', 'std', lambda x: (x - x.median()).abs().median()],
                                                      'error_diff': ['mean', 'std']
                                                      })
  
  avg_summarized_squared_errors[lag].columns = ['_'.join(col).strip() for col in avg_summarized_squared_errors[lag].columns.values]
  avg_summarized_squared_errors[lag].rename(columns={
      'full_squared_errors_<lambda_0>': 'full_squared_errors_mad',
      'restr_squared_errors_<lambda_0>': 'restr_squared_errors_mad'
  }, inplace=True)

  
  avg_summarized_squared_errors[lag].columns = avg_summarized_squared_errors[lag].columns.str.replace('_first', '')
  avg_summarized_squared_errors[lag] = avg_summarized_squared_errors[lag].set_index('dvg', drop=True)
  
  print(f"Mean summarized squared errors for lag {lag}:")
  display(avg_summarized_squared_errors[lag].sort_values('full_squared_errors_mean', ascending=True))
  summarized_fold_maes[lag]['grouping_id'] = summarized_fold_maes[lag]['dvg'].apply(lambda x: x.split('_')[0] if x.startswith('restr_') else x)
  avg_summarized_maes[lag] = summarized_fold_maes[lag].groupby(['grouping_id']).agg({
                                                      'dvg': 'first',
                                                      'label': 'first',
                                                      'ssr_chi2test_pval': 'first',
                                                      'dpi': 'count',
                                                      'full_mae': ['mean', 'median', 'std', lambda x: (x - x.median()).abs().median()],
                                                      'restr_mae': ['mean', 'median', 'std', lambda x: (x - x.median()).abs().median()],
                                                      'mae_diff': ['mean', 'std']
                                                      })
  avg_summarized_maes[lag].columns = ['_'.join(col).strip() for col in avg_summarized_maes[lag].columns.values]
  avg_summarized_maes[lag].rename(columns={
      'full_mae_<lambda_0>': 'full_mae_mad',
      'restr_mae_<lambda_0>': 'restr_mae_mad',
  }, inplace=True)
  
  
  avg_summarized_maes[lag].columns = avg_summarized_maes[lag].columns.str.replace('_first', '')
  avg_summarized_maes[lag] = avg_summarized_maes[lag].set_index('dvg', drop=True)
  
  print(f"Mean summarized MAEs for lag {lag}:")
  display(avg_summarized_maes[lag].sort_values('full_mae_mean', ascending=True))

  median_summarized_maes[lag] = avg_summarized_maes[lag].copy()
  
  mean_stats_tests_dfs[lag] = pairwise_stats_tests(
      df=avg_summarized_squared_errors[lag].reset_index(),
      value_col='full_squared_errors_mean',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  for stat_test in mean_stats_tests_dfs[lag]:
    print(f"Mean stats test '{stat_test}' for lag {lag}:")
    display(mean_stats_tests_dfs[lag][stat_test])
    
  median_stats_tests_dfs[lag] = pairwise_stats_tests(
      df=avg_summarized_squared_errors[lag].reset_index(),
      value_col='full_squared_errors_median',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  for stat_test in median_stats_tests_dfs[lag]:
    print(f"Median stats test '{stat_test}' for lag {lag}:")
    display(median_stats_tests_dfs[lag][stat_test])

  mean_mae_stats_tests_dfs[lag] = pairwise_stats_tests(
      df=avg_summarized_maes[lag].reset_index(),
      value_col='full_mae_mean',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  for stat_test in mean_mae_stats_tests_dfs[lag]:
    print(f"Mean MAE stats test '{stat_test}' for lag {lag}:")
    display(mean_mae_stats_tests_dfs[lag][stat_test])

  median_mae_stats_tests_dfs[lag] = pairwise_stats_tests(
      df=avg_summarized_maes[lag].reset_index(),
      value_col='full_mae_median',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  for stat_test in median_mae_stats_tests_dfs[lag]:
    print(f"Median MAE stats test '{stat_test}' for lag {lag}:")
    display(median_mae_stats_tests_dfs[lag][stat_test])

In [ ]:
! mkdir -p {output_prefix}/plots

for lag in range(1, 2):
  df = avg_summarized_squared_errors[lag]

  df['non_zero_counts'] = (
      df.index.astype(str)
        .to_series(index=df.index)
        .map(dvg2nonzero_counts)
        .fillna(df['ssr_chi2test_pval'])
  )

  r = df.groupby('label')['non_zero_counts'].rank(method='dense', ascending=True)

  k = df.groupby('label')['non_zero_counts'].transform(lambda s: s.rank(method='dense').nunique())
  df['non_zero_counts_norm'] = np.where(k > 1, (r - 1) / (k - 1), 0.0)

  df['non_zero_counts_norm'] = df['non_zero_counts_norm'].fillna(0.0)
  fig, ax = make_performance_swarmplot(
      df.copy().reset_index(),
      ref_performance_dct={"FULL_SQUARED_ERRORS_MEAN": avg_summarized_squared_errors[lag]['restr_squared_errors_mean'].mean()},
      performance_metric='full_squared_errors_mean',
      forecasted_value='pfu',
      title=f'Granger-related DVGs predictive power over PFU (Mean over folds, lag={lag})\n',
      ylabel='Mean Squared Error',
      dot_color_ref='non_zero_counts_norm',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=25,
      ymin=-10,
      p_val_pos=20,
      figsize=(12,6),
      dotsize=5,
      linewidth=2,
      stats_matrix=mean_stats_tests_dfs[lag].get('mannwhitney_p'),
      stat_test_name='mannwhitney_p',
      effect_size_matrix=mean_stats_tests_dfs[lag].get('cliffs_delta'),
      effect_size_name='cliffs_delta',
      alpha=0.05,
      order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped']
  )
  xmin, xmax = ax.get_xlim()
  ax.hlines(
        y=avg_summarized_squared_errors[lag]['restr_squared_errors_mean'].mean(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=2, zorder=10
    )
  ax.hlines(
        y=min(avg_summarized_squared_errors[lag]['restr_squared_errors_mean'].mean() - \
          avg_summarized_squared_errors[lag]['restr_squared_errors_std'].mean(),
          0),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  ax.hlines(
        y=avg_summarized_squared_errors[lag]['restr_squared_errors_mean'].mean() + \
          avg_summarized_squared_errors[lag]['restr_squared_errors_std'].mean(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  
  fig.savefig(f"{output_prefix}/plots/mean_swarm_performance_lag{lag}.png", dpi=300)

In [ ]:
! mkdir -p {output_prefix}/plots
for lag in range(1, 2):
  df = avg_summarized_squared_errors[lag]

  df['non_zero_counts'] = (
      df.index.astype(str)
        .to_series(index=df.index)
        .map(dvg2nonzero_counts)
        .fillna(df['ssr_chi2test_pval'])
  )

  r = df.groupby('label')['non_zero_counts'].rank(method='dense', ascending=True)

  k = df.groupby('label')['non_zero_counts'].transform(lambda s: s.rank(method='dense').nunique())
  df['non_zero_counts_norm'] = np.where(k > 1, (r - 1) / (k - 1), 0.0)

  df['non_zero_counts_norm'] = df['non_zero_counts_norm'].fillna(0.0)
  fig, ax = make_performance_swarmplot(
      df.copy().reset_index(),
      ref_performance_dct={"FULL_SQUARED_ERRORS_MEDIAN": avg_summarized_squared_errors[lag]['restr_squared_errors_median'].median()},
      performance_metric='full_squared_errors_median',
      forecasted_value='pfu',
      title=f'Granger-related DVGs predictive power over PFU (Median over folds, lag={lag})\n',
      ylabel='Median Squared Error',
      dot_color_ref='non_zero_counts_norm',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=25,
      ymin=-10,
      p_val_pos=20,
      figsize=(12,6),
      dotsize=5,
      linewidth=2,
      stats_matrix=median_stats_tests_dfs[lag][24].get('mannwhitney_p'),
      stat_test_name='mannwhitney_p',
      effect_size_matrix=median_stats_tests_dfs[lag][24].get('cliffs_delta'),
      effect_size_name='cliffs_delta',
      alpha=0.05,
      order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped']
  )
  xmin, xmax = ax.get_xlim()
  ax.hlines(
        y=avg_summarized_squared_errors[lag]['restr_squared_errors_median'].median(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=2, zorder=10
    )
  ax.hlines(
        y=min(avg_summarized_squared_errors[lag]['restr_squared_errors_median'].median() - \
          avg_summarized_squared_errors[lag]['restr_squared_errors_mad'].median(),
          0),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  ax.hlines(
        y=avg_summarized_squared_errors[lag]['restr_squared_errors_median'].median() + \
          avg_summarized_squared_errors[lag]['restr_squared_errors_mad'].median(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  
  fig.savefig(f"{output_prefix}/plots/median_swarm_performance_lag{lag}.png", dpi=300)

In [ ]:
! mkdir -p {output_prefix}/plots
for lag in range(1, 2):
  df = avg_summarized_maes[lag]

  df['non_zero_counts'] = (
      df.index.astype(str)
        .to_series(index=df.index)
        .map(dvg2nonzero_counts)
        .fillna(1)
  )

  r = df.groupby('label')['non_zero_counts'].rank(method='dense', ascending=True)

  k = df.groupby('label')['non_zero_counts'].transform(lambda s: s.rank(method='dense').nunique())
  df['non_zero_counts_norm'] = np.where(k > 1, (r - 1) / (k - 1), 0.0)

  df['non_zero_counts_norm'] = df['non_zero_counts_norm'].fillna(0.0)
    
  fig, ax = make_performance_swarmplot(
      df.copy().reset_index(),
      ref_performance_dct={"FULL_MAE_MEDIAN": avg_summarized_maes[lag]['restr_mae_median'].median()},
      performance_metric='full_mae_median',
      forecasted_value='pfu',
      title=f'Granger-related DVGs predictive power over PFU (Median over folds, lag={lag})\n',
      ylabel='MAE',
      dot_color_ref='non_zero_counts_norm',
      inverted_colors=True,
      upper_border=-0.05,
      ymax=4,
      ymin=-2,
      p_val_pos=3,
      figsize=(12,6),
      dotsize=3,
      linewidth=2,
      stats_matrix=median_stats_tests_dfs[lag][24].get('mannwhitney_p'),
      stat_test_name='mannwhitney_p',
      effect_size_matrix=median_stats_tests_dfs[lag][24].get('cliffs_delta'),
      effect_size_name='cliffs_delta',
      alpha=0.05,
      order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped']
  )
  xmin, xmax = ax.get_xlim()
  ax.hlines(
        y=avg_summarized_maes[lag]['restr_mae_median'].median(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=2, zorder=10
    )
  ax.hlines(
        y=max(avg_summarized_maes[lag]['restr_mae_median'].median() - \
          avg_summarized_maes[lag]['restr_mae_mad'].median(),
          0),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  ax.hlines(
        y=avg_summarized_maes[lag]['restr_mae_median'].median() + \
          avg_summarized_maes[lag]['restr_mae_mad'].median(),
        xmin=xmin, xmax=xmax,
        label='restricted model performance',
        color='#e09312', linewidth=1, zorder=10
    )
  
  fig.savefig(f"{output_prefix}/plots/median_swarm_performance_mae_lag{lag}.png", dpi=300)

In [ ]:
plt.rcParams.update({'font.size': 14})
bh_corrected_label_dct = bh_squared_errors_df[1][1].set_index('dvg')['label'].to_dict()
avg_summarized_squared_errors_corrected = {}
avg_summarized_maes_corrected = {}
median_stats_tests_dfs_corrected = {}
mean_stats_tests_dfs_corrected = {}
bh_corrected_label_dct['restricted'] = 'restricted'

for lag in range(1, 2):
  avg_summarized_squared_errors_corrected[lag] = avg_summarized_squared_errors[lag].copy()
  avg_summarized_squared_errors_corrected[lag]['label'] = avg_summarized_squared_errors_corrected[lag].index.map(bh_corrected_label_dct)
  avg_summarized_squared_errors_corrected[lag]['label'] = avg_summarized_squared_errors_corrected[lag]['label'].fillna('restricted')
  
  avg_summarized_maes_corrected[lag] = avg_summarized_maes[lag].copy()
  avg_summarized_maes_corrected[lag]['label'] = avg_summarized_maes_corrected[lag].index.map(bh_corrected_label_dct)
  avg_summarized_maes_corrected[lag]['label'] = avg_summarized_maes_corrected[lag]['label'].fillna('restricted')
  
  median_stats_tests_dfs_corrected[lag] = pairwise_stats_tests(
      df=avg_summarized_maes_corrected[lag].reset_index(),
      value_col='full_mae_median',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  mean_stats_tests_dfs_corrected[lag] = pairwise_stats_tests(
      df=avg_summarized_maes_corrected[lag].reset_index(),
      value_col='full_mae_mean',
      group_col='label',
      dpi_col='dpi_count',
      mw_adjust='bonferroni',
      label_order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped'],
      dunn_adjust='bonferroni'
  )
  
  df_sqe = avg_summarized_squared_errors_corrected[lag]

  df_sqe['non_zero_counts'] = (
      df_sqe.index.astype(str)
        .to_series(index=df_sqe.index)
        .map(dvg2nonzero_counts)
        .fillna(1.0)
  )
  grouped = df_sqe.groupby('label')
  df_sqe['group_norm_ssr_chi2test_pval'] = df_sqe['ssr_chi2test_pval']
  for label, group in grouped:
    max_value = group['group_norm_ssr_chi2test_pval'].max()
    df_sqe.loc[group.index, 'group_norm_ssr_chi2test_pval'] = group['group_norm_ssr_chi2test_pval'] / max_value
  r = df_sqe.groupby('label')['non_zero_counts'].rank(method='dense', ascending=True)

  k = df_sqe.groupby('label')['non_zero_counts'].transform(lambda s: s.rank(method='dense').nunique())
  df_sqe['non_zero_counts_norm'] = np.where(k > 1, (r - 1) / (k - 1), 0.0)

  df_sqe['non_zero_counts_norm'] = df_sqe['non_zero_counts_norm'].fillna(0.0)
  
  df_sqe['ssr_chi2test_pval_norm'] = df_sqe['ssr_chi2test_pval'].rank(method='dense', ascending=False) / df_sqe['ssr_chi2test_pval'].rank(method='dense', ascending=True).max()
  df_sqe['ssr_chi2test_pval_norm'] = df_sqe['ssr_chi2test_pval_norm'].fillna(1.0)
  
  df_mae = avg_summarized_maes_corrected[lag]
  df_mae['non_zero_counts'] = (
      df_mae.index.astype(str)
        .to_series(index=df_mae.index)
        .map(dvg2nonzero_counts)
        .fillna(1.0)
  )

  grouped = df_mae.groupby('label')
  df_mae['group_norm_ssr_chi2test_pval'] = df_mae['ssr_chi2test_pval']
  for label, group in grouped:
    max_value = group['group_norm_ssr_chi2test_pval'].max()
    df_mae.loc[group.index, 'group_norm_ssr_chi2test_pval'] = group['group_norm_ssr_chi2test_pval'] / max_value
  
  r = df_mae.groupby('label')['non_zero_counts'].rank(method='dense', ascending=True)
  k = df_mae.groupby('label')['non_zero_counts'].transform(lambda s: s.rank(method='dense').nunique())
  df_mae['non_zero_counts_norm'] = np.where(k > 1, (r - 1) / (k - 1), 0.0)
  df_mae['non_zero_counts_norm'] = df_mae['non_zero_counts_norm'].fillna(1.0)
  
  df_mae['ssr_chi2test_pval_norm'] = df_mae['ssr_chi2test_pval'].rank(method='dense', ascending=False) / df_mae['ssr_chi2test_pval'].rank(method='dense', ascending=True).max()
  df_mae['ssr_chi2test_pval_norm'] = df_mae['ssr_chi2test_pval_norm'].fillna(1.0)
  
  
  # fig1, ax1 = make_performance_swarmplot(
  #     df_sqe.copy().reset_index(),
  #     ref_performance_dct={"FULL_SQUARED_ERRORS_MEAN": avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mean'].mean()},
  #     performance_metric='full_squared_errors_mean',
  #     forecasted_value='pfu',
  #     title=f'Granger-related DVGs predictive power over PFU (Mean over folds, lag={lag})\n',
  #     ylabel='Mean Squared Error',
  #     dot_color_ref='non_zero_counts_norm',
  #     inverted_colors=True,
  #     upper_border=-0.05,
  #     ymax=20,
  #     ymin=-10,
  #     p_val_pos=15,
  #     figsize=(12,6),
  #     dotsize=5,
  #     linewidth=2,
  #     #stats_matrix=mean_stats_tests_dfs_corrected[lag].get('mannwhitney_p'),
  #     #stat_test_name='mannwhitney_p',
  #     #effect_size_matrix=mean_stats_tests_dfs_corrected[lag].get('cliffs_delta'),
  #     #effect_size_name='cliffs_delta',
  #     alpha=0.05,
  #     order=['restricted','causing', 'bi-directional', 'caused', 'non-related', 'shuffled', 'bootstrapped']
  #   )
  # xmin, xmax = ax1.get_xlim()
  # ax1.hlines(
  #       y=avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mean'].mean(),
  #       xmin=xmin, xmax=xmax,
  #       label='restricted model performance',
  #       color='#e09312', linewidth=2, zorder=10
  #   )
  # ax1.hlines(
  #       y=min(avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mean'].mean() - \
  #         avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_std'].mean(),
  #         0),
  #       xmin=xmin, xmax=xmax,
  #       label='restricted model performance',
  #       color='#e09312', linewidth=1, zorder=10
  #   )
  # ax1.hlines(
  #       y=avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mean'].mean() + \
  #         avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_std'].mean(),
  #       xmin=xmin, xmax=xmax,
  #       label='restricted model performance',
  #       color='#e09312', linewidth=1, zorder=10
  #   )
  # fig1.savefig(f"{output_prefix}/plots/mean_swarm_performance_lag{lag}_bh_corrected.png", dpi=300)
  
  fig2, ax2 = make_performance_swarmplot(
    df_mae.copy().reset_index(),
    ref_performance_dct={"FULL_MAE_MEDIAN": avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_median'].median()},
    performance_metric='full_mae_median',
    forecasted_value='pfu',
    title=f"Median MAE of full models by Granger-causality label (lag=1)\nPairwise Mann–Whitney U (Bonferroni-corrected), Cliff's δ shown for p < 0.001\n",
    ylabel='MAE',
    dot_color_ref='group_norm_ssr_chi2test_pval',
    inverted_colors=True,
    upper_border=-0.05,
    ymax=4,
    ymin=0,
    p_val_pos=3,
    figsize=(12,6),
    dotsize=3,
    linewidth=2,
    stats_matrix=median_stats_tests_dfs_corrected[lag][24].get('mannwhitney_p'),
    stat_test_name='mannwhitney_p',
    effect_size_matrix=median_stats_tests_dfs_corrected[lag][24].get('cliffs_delta'),
    effect_size_name='cliffs_delta',
    alpha=0.001,
    order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled'],
    plot_reference=False
  )
  xmin, xmax = ax2.get_xlim()
  ax2.hlines(
        y=avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_median'].median(),
        xmin=xmin, xmax=xmax,
        label='restricted\nmodel\nmedian',
        color='#e09312', linewidth=2, zorder=10
    )
  ax2.hlines(
        y=max(avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_median'].median() - \
          avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_mad'].median(),
          0),
        xmin=xmin, xmax=xmax,
        label='restricted\nmodel\nMAD',
        color='#e09312', linewidth=1, zorder=10
    )
  ax2.hlines(
        y=avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_median'].median() + \
          avg_summarized_maes_corrected[lag][avg_summarized_maes_corrected[lag]['label'] == 'restricted']['full_mae_mad'].median(),
        xmin=xmin, xmax=xmax,
        color='#e09312', linewidth=1, zorder=10
    )
  ax2.legend(loc='center left', bbox_to_anchor=(1, 0.5))
  fig2.tight_layout()
  fig2.savefig(f"{output_prefix}/plots/median_swarm_performance_mae_lag{lag}_bh_corrected.png", dpi=300)
  
  # fig3, ax3 = make_performance_swarmplot(
  #   df_sqe.copy().reset_index(),
  #   ref_performance_dct={"FULL_SQUARED_ERRORS_MEDIAN": avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_median'].median()},
  #   performance_metric='full_squared_errors_median',
  #   forecasted_value='pfu',
  #   title=f'Granger-related DVGs predictive power over PFU (Median over folds, lag={lag})\n',
  #   ylabel='Median Squared Error',
  #   dot_color_ref='group_norm_ssr_chi2test_pval',
  #   inverted_colors=True,
  #   upper_border=-0.05,
  #   ymax=25,
  #   ymin=-10,
  #   p_val_pos=20,
  #   figsize=(12,6),
  #   dotsize=5,
  #   linewidth=2,
  #   stats_matrix=median_stats_tests_dfs_corrected[lag][24].get('mannwhitney_p'),
  #   stat_test_name='mannwhitney_p',
  #   effect_size_matrix=median_stats_tests_dfs_corrected[lag][24].get('cliffs_delta'),
  #   effect_size_name='cliffs_delta',
  #   alpha=0.001,
  #   order=['causing', 'bi-directional', 'caused', 'non-related', 'shuffled'],
  #   plot_reference=False
  # )
  # xmin, xmax = ax3.get_xlim()
  # ax3.hlines(
  #       y=avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_median'].median(),
  #       xmin=xmin, xmax=xmax,
  #       label='median restricted model',
  #       color='#e09312', linewidth=2, zorder=10
  #   )
  # ax3.hlines(
  #       y=max(avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_median'].median() - \
  #         avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mad'].median(),
  #         0),
  #       xmin=xmin, xmax=xmax,
  #       label='restricted model MAD',
  #       color='#e09312', linewidth=1, zorder=10
  #   )
  # ax3.hlines(
  #       y=avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_median'].median() + \
  #         avg_summarized_squared_errors_corrected[lag][avg_summarized_squared_errors_corrected[lag]['label'] == 'restricted']['full_squared_errors_mad'].median(),
  #       xmin=xmin, xmax=xmax,
  #       color='#e09312', linewidth=1, zorder=10
  #   )
  # # bottom right
  # ax3.legend(loc='center left', bbox_to_anchor=(1, 0.5))
  # fig3.tight_layout()
  # fig3.savefig(f"{output_prefix}/plots/median_swarm_performance_lag{lag}_bh_corrected_v2.png", dpi=300)

In [ ]:
avg_summarized_maes_corrected[1][avg_summarized_maes_corrected[1].index.isin(['PB2_217_2204', 'PB2_269_2202', 'PB2_129_2176','restr_9.42'])][['full_mae_median', 'full_mae_mad']]

In [ ]:
import statsmodels
import pandas
import matplotlib

print(f"statsmodels: {statsmodels.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"matplotlib: {matplotlib.__version__}")